# 決策樹（1986）

- **核心原理**：資訊增益／Gini 分裂
- **變體**：ID3（資訊增益、多元分裂、只吃類別特徵）→ C4.5（增益比、連續特徵、剪枝）→ CART（二元分裂、Gini、成本複雜度剪枝；sklearn 用的就是 CART）
- **推薦資料集**：Titanic（1,309 位乘客；目標：是否生還）

本章依照 README 的「每章標準流程」：EDA → Baseline → 手寫模型 → 套件實作 → 調參 → 測試集評估 → 診斷 → 本章小結。

決策樹和前四章都不一樣：

- 前三章（感知器、線性回歸、邏輯回歸）學一組**係數** $w$，用梯度或閉式解求出來；
- 第四章 KNN 不學參數，靠**距離**；
- 決策樹不用梯度、也不用距離，而是**一連串的是非題**（「是男性嗎？」「艙等 ≤ 2 嗎？」），每一題都用「貪婪搜尋」從所有特徵、所有切點裡挑出「切完之後最純」的那一刀。

所以這一章的重點是：**什麼叫做「純」（不純度）、怎麼有效率地找最好的切點、樹長多深才不會背答案。**

### 閱讀指南：本章的符號與跨章引用

下一格是「前置複習」；第 2、3A、3B、3C、3D、4、5、7 節開頭另有各自的符號表或名詞表。

**資料**：每一列是一位乘客。目標 `survived`：1 = 生還、0 = 罹難。本章用到的特徵：

| 欄位 | 意義 | 型態 |
| --- | --- | --- |
| `pclass` | 艙等：1 = 頭等、2 = 二等、3 = 三等 | 整數（有順序的類別） |
| `sex` | 性別 | 類別（female／male） |
| `age` | 年齡（歲；嬰兒是小數，例如 0.92） | 連續，有缺值 |
| `sibsp` | 同船的兄弟姊妹＋配偶人數 | 計數 |
| `parch` | 同船的父母＋子女人數 | 計數 |
| `fare` | 票價（英鎊） | 連續，極度右偏 |
| `embarked` | 登船港口：C = Cherbourg、Q = Queenstown、S = Southampton | 類別 |

**本章最常出現的符號**（各小節開頭有完整符號表）：

| 符號 | 意義 | 本章的值 |
| --- | --- | --- |
| $N$ | 訓練樣本總數 | 1,047（以載入結果為準） |
| $d$ | 前處理後的特徵數 | 9 |
| $K$ | 類別數 | 2 |
| $t$ | 樹上的一個**節點**（node）；$t_L$、$t_R$ 是它的左、右子節點 | — |
| $n_t$ | 落在節點 $t$ 的訓練樣本數 | — |
| $p_k(t)$ | 節點 $t$ 裡第 $k$ 類的比例 | — |
| $I(t)$ | 節點 $t$ 的**不純度**（impurity）：Gini 或 entropy | — |
| $(j, s)$ | 一個**切法**：第 $j$ 個特徵、門檻 $s$；$x_j \le s$ 走左邊、否則走右邊 | — |
| $\Delta I$ | 不純度下降量：$I(t) - \frac{n_L}{n_t} I(t_L) - \frac{n_R}{n_t} I(t_R)$ | — |
| 深度（depth） | 從根節點走到某節點要經過幾次切分；根節點深度 0 | — |
| 葉子（leaf） | 不再切分的節點；預測 = 葉子裡的類別比例 | — |
| $\alpha$ | 成本複雜度剪枝的懲罰係數（`ccp_alpha`） | — |

**跨章引用對照**：

| 文中寫法 | 指的是 | 那裡的結論（重述） |
| --- | --- | --- |
| 第三章 | 第一部分第 3 節：邏輯回歸（`01-classic-ml/03-logistic-regression`），Breast Cancer | 見下面幾列。 |
| 第三章「log loss baseline = 熵」 | 同上，TODO 2 | 「不看特徵、只輸出類別比例」的 log loss = 標籤分布的熵 $-\sum_k q_k \ln q_k$；二元時最大是 $\ln 2 \approx 0.693$。 |
| 第三章「標準化是必要的」 | 同上 | 梯度下降時尺度差異讓收斂極慢；L2 懲罰對不同尺度的係數不公平。 |
| 第三章 C 的 validation curve | 同上 | C 越大 = 模型越複雜；過擬合在圖的**右邊**。 |
| 第四章 | 第一部分第 4 節：KNN（`01-classic-ml/04-knn`），Digits | 見下面幾列。 |
| 第四章「縮放 = 距離的權重」 | 同上，TODO 11 | KNN 的每個特徵佔平方距離的份量 $\propto$ 縮放後的變異，所以標準化會直接改變誰是鄰居（Digits 上 LOO 0.988 → 0.974）。 |
| 第四章「$k$ 小 = 複雜」 | 同上，TODO 10、14 | KNN 的複雜度與 $k$ 反方向：$k$ 小時邊界跟著個別點走（過擬合在**左邊**）。 |
| 第四章的平手 | 同上，TODO 8 | 整數特徵讓距離常常一樣；平手時誰排前面取決於實作，比的應該是結果而不是編號。 |
| 第四章的 1-SE | 同上，TODO 15、16 | 跟 CV 最佳差不到一個標準差的候選中挑「最簡單」的；但要看「簡單」換到了什麼實際好處。 |
| 第 10 章 Random Forest | 第一部分第 10 節（`01-classic-ml/10-random-forest`） | 很多棵在 bootstrap 樣本上長的樹平均起來；手寫版會以本章的決策樹為基底。 |
| 第 12 章 GBDT | 第一部分第 12 節 | 一棵接一棵的小樹，每棵擬合前面的殘差。 |

### 前置複習：本章會用到的舊知識

#### 數學

**1. 比例與機率**：一個節點有 $n_t$ 筆，其中 $n_{t,k}$ 筆是第 $k$ 類，$p_k(t) = n_{t,k} / n_t$，$\sum_k p_k(t) = 1$。例：10 筆、3 筆生還 → $p_1 = 0.3$、$p_0 = 0.7$。

**2. 熵（entropy）**：$H(p) = -\sum_k p_k \log_2 p_k$（以 2 為底，單位是 bit）。約定 $0 \cdot \log 0 = 0$（$p \to 0$ 時 $p\log p \to 0$）。

| $p = (p_0, p_1)$ | $H$ | 直覺 |
| --- | --- | --- |
| $(0.5, 0.5)$ | 1 | 最不確定：猜生還或罹難都一樣沒把握 |
| $(0.9, 0.1)$ | 0.469 | 大致確定 |
| $(1, 0)$ | 0 | 完全確定（純的） |

換底：$\log_2 x = \ln x / \ln 2$。第三章 log loss 用的是自然對數 $\ln$，數值是 bit 的 $\ln 2 \approx 0.693$ 倍。

**3. Gini 不純度**：$G(p) = 1 - \sum_k p_k^2 = \sum_k p_k (1 - p_k)$。意義：從節點裡**隨機抽一筆、再依比例隨機貼一個標籤**，貼錯的機率。例：$(0.5, 0.5)$ → $1 - 0.25 - 0.25 = 0.5$（二元的最大值）；$(0.9, 0.1)$ → $1 - 0.81 - 0.01 = 0.18$。

**4. 加權平均**：兩組人數 $n_L$、$n_R$，各有一個數值 $a_L$、$a_R$，合起來的平均是 $\frac{n_L}{n_L + n_R} a_L + \frac{n_R}{n_L + n_R} a_R$。例：左邊 2 人平均 10、右邊 8 人平均 20 → $0.2 \cdot 10 + 0.8 \cdot 20 = 18$。

**5. 凹函數（concave）**：圖形「往上拱」的函數，例如 $\sqrt{x}$、$-x^2$。性質（Jensen 不等式）：**函數在「平均」的值 $\ge$ 函數值的平均**，$f(\lambda a + (1-\lambda) b) \ge \lambda f(a) + (1-\lambda) f(b)$。例：$f(x) = \sqrt{x}$、$a = 0$、$b = 4$、$\lambda = 0.5$ → 左邊 $\sqrt{2} \approx 1.41$ $\ge$ 右邊 $0.5 \cdot 0 + 0.5 \cdot 2 = 1$。（TODO 12 會用到：Gini 和 entropy 都是凹函數。）

**6. 單調遞增轉換**：$a < b \Rightarrow f(a) < f(b)$（例如 $\log$、$\sqrt{\ }$、$ax + c$ 且 $a > 0$）。只改變數值的「間距」，不改變**排序**。例：$(1, 10, 100)$ 取 $\log_{10}$ → $(0, 1, 2)$，順序不變。

**7. Bootstrap 抽樣**：從 $N$ 筆資料中「抽後放回」抽 $N$ 次。每一筆沒被抽中的機率 $= (1 - 1/N)^N \approx e^{-1} \approx 0.368$ → 每個 bootstrap 樣本大約只包含 63.2% 的不同資料，其餘是重複的。

**8. 偏差—變異**（第二、三、四章的重述）：偏差高 = 模型太簡單、連訓練資料都學不好（欠擬合）；變異高 = 模型對「剛好拿到哪些訓練資料」很敏感，換一批資料結果就大變（過擬合）。

**9. Big-O**：$O(f)$ = 成本大約與 $f$ 成正比。排序 $n$ 個數是 $O(n \log n)$；走過一次是 $O(n)$。

#### 程式

**10. 遞迴（recursion）**：函數呼叫自己，每次處理「比較小」的問題，直到碰到**終止條件**。

```python
def depth_of(n):          # 把 n 一直對半切，切幾次才剩 1？
    if n <= 1:            # 終止條件：不能再切
        return 0
    return 1 + depth_of(n // 2)   # 遞迴：處理一半，加上「這一刀」
depth_of(8)               # 3   （8 → 4 → 2 → 1）
```

決策樹的 `_grow(idx, depth)` 也是這樣：「長一棵樹」= 「切一刀」+「左邊長一棵樹」+「右邊長一棵樹」。

**11. `@dataclass` 與物件參照**：`@dataclass` 自動產生 `__init__`，欄位寫在 class 裡就好。屬性可以存**另一個物件**（不是複製），`None` 表示「沒有」。

```python
from dataclasses import dataclass

@dataclass
class Box:
    value: int
    child: "Box | None" = None   # 預設沒有子節點；型別寫成字串是因為 Box 還沒定義完

a = Box(1, child=Box(2))
a.child.value       # 2：a.child 本身就是一個 Box
a.child.child       # None
```

**12. 一起排序兩個陣列**：`order = np.argsort(x)` 回傳「讓 x 由小到大的位置」，`x[order]`、`y[order]` 用**同一組位置**重排，兩者仍一一對應。

```python
x = np.array([30, 10, 20]); y = np.array([1, 0, 1])
order = np.argsort(x, kind="stable")   # [1, 2, 0]
x[order], y[order]                     # [10, 20, 30], [0, 1, 1]
```

**13. 累加 `np.cumsum`**：`np.cumsum([1, 0, 1, 1])` → `[1, 1, 2, 3]`，第 $i$ 個 = 前 $i+1$ 個的和。二維時 `axis=0` 是「沿著列往下累加，每一欄各自加」：

```python
A = np.array([[1, 0], [0, 1], [1, 0]])
np.cumsum(A, axis=0)   # [[1, 0], [1, 1], [2, 1]]
```

**14. 布林遮罩分組**：`mask = x <= 2.5`；`idx[mask]` 是符合的、`idx[~mask]` 是不符合的（`~` 是 NOT）。

**15. one-hot 與 `np.eye(K)[y]`**：`np.eye(2)[[0, 1, 1]]` → `[[1, 0], [0, 1], [0, 1]]`，每一列只有自己類別那一格是 1；對它沿 `axis=0` 加總 = 每一類的筆數（等於 `np.bincount(y, minlength=K)`）。

**16. `np.divide(a, b, out=..., where=...)`**：只在 `where` 為 True 的位置做除法，其餘保留 `out` 的值 → 避免 $0/0$ 的警告。`np.where(cond, A, B)` 則是兩邊都先算好再挑，`A` 裡的 $\log 0$ 仍會先算出來（會跳警告但結果正確）。

**17. pandas**：`df.groupby("sex")["survived"].mean()` = 每組的生還率；`pd.crosstab(a, b, values=y, aggfunc="mean")` = 二維分組的平均；`pd.cut(age, bins=[0, 12, 18, ...])` 把連續值切成區間。

---
## 1. 資料與探索性資料分析（EDA）

觀察直接寫在每格程式最上面的註解（`# 觀察：`）。每格都附了「看什麼」的提示，請把看到的數字與結論寫進去。

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from ml_dl_llm.data import load_titanic

X_train, X_test, y_train, y_test = load_titanic()
# 目標轉成整數 0／1（OpenML 的 survived 是字串類別 "0"／"1"）
y_train = y_train.astype(int)
y_test = y_test.astype(int)
train = X_train.assign(survived=y_train)

print("train / test：", X_train.shape, X_test.shape)
print("欄位：", X_train.columns.tolist())
display(X_train.head())

In [ ]:
# 型態與缺值
# 觀察：（哪些欄位是數值、哪些是文字／類別？哪幾欄有缺值、缺多少比例？）
info = pd.DataFrame(
    {
        "dtype": X_train.dtypes.astype(str),
        "缺值比例": X_train.isna().mean().round(3),
        "不同值個數": X_train.nunique(),
    }
)
display(info)

In [ ]:
# 類別分布
# 觀察：（生還比例多少？「全部猜罹難」的 accuracy 大約多少？train／test 比例一致嗎？）
display(
    pd.DataFrame(
        {
            "train": y_train.value_counts(normalize=True).sort_index(),
            "test": y_test.value_counts(normalize=True).sort_index(),
        }
    ).round(3)
)

In [ ]:
# 「事後才知道」的欄位：boat（搭上的救生艇編號）、body（遺體編號）
# 觀察：（有 boat 的人生還率多少？有 body 的人呢？這兩欄能拿來當特徵嗎？TODO 1 (1)）
for col in ["boat", "body", "home.dest", "cabin"]:
    if col in train:
        g = train.groupby(train[col].notna())["survived"].agg(["mean", "size"])
        g.index = [f"{col} 缺值", f"{col} 有值"]
        display(g.round(3))

In [ ]:
# 每個特徵和生還率的關係（類別型特徵）
# 觀察：（哪一個特徵單獨就把生還率拉得最開？女性 vs 男性、頭等 vs 三等各是多少？
#        sex × pclass 的交叉表裡，有沒有「某個特徵的效果在另一個特徵的不同組別裡不一樣」？）
fig, axes = plt.subplots(1, 3, figsize=(13, 3.5))
for ax, col in zip(axes, ["sex", "pclass", "embarked"]):
    rate = train.groupby(col, observed=True)["survived"].agg(["mean", "size"])
    ax.bar(rate.index.astype(str), rate["mean"], color="#4C72B0")
    for i, (m, s) in enumerate(zip(rate["mean"], rate["size"])):
        ax.text(i, m, f"{m:.2f}\n(n={s})", ha="center", va="bottom", fontsize=9)
    ax.set_ylim(0, 1.1)
    ax.set_title(f"survival rate by {col}")
fig.tight_layout()
plt.show()

print("sex × pclass 的生還率：")
display(
    pd.crosstab(train["sex"], train["pclass"], values=train["survived"], aggfunc="mean").round(3)
)
print("sex × pclass 的人數：")
display(pd.crosstab(train["sex"], train["pclass"]))

In [ ]:
# 連續／計數特徵：age、fare、sibsp、parch
# 觀察：（年齡多小的孩子生還率明顯較高？票價的分布長什麼樣子（看最大值與中位數）？
#        age 缺值的人生還率和有年齡的人一樣嗎？——缺值本身可能就帶有資訊）
fig, axes = plt.subplots(1, 2, figsize=(13, 3.5))
for ax, col, bins in [(axes[0], "age", 40), (axes[1], "fare", 60)]:
    for lab, name in [(0, "died"), (1, "survived")]:
        ax.hist(train.loc[train["survived"] == lab, col].dropna(), bins=bins, alpha=0.6, label=name)
    ax.set_title(col)
    ax.legend()
plt.show()

age_bins = pd.cut(train["age"], bins=[0, 5, 12, 18, 30, 45, 60, 80])
display(train.groupby(age_bins, observed=True)["survived"].agg(["mean", "size"]).round(3))
print("age 缺值者的生還率：", round(train.loc[train["age"].isna(), "survived"].mean(), 3),
      "；有年齡者：", round(train.loc[train["age"].notna(), "survived"].mean(), 3))
print("fare：中位數", train["fare"].median(), "、平均", round(train["fare"].mean(), 1),
      "、最大", train["fare"].max())

family = train["sibsp"] + train["parch"] + 1
display(train.groupby(family.rename("family_size"))["survived"].agg(["mean", "size"]).round(3).T)

In [ ]:
# TODO 1（問答）EDA 小結
#
# (1) boat 與 body 的生還率幾乎是 0 或 1。為什麼這兩欄不能當特徵？
#     這種問題叫什麼？（提示：預測「會不會生還」的時間點，能不能知道他搭上哪一艘救生艇？）
#     home.dest、name、ticket 也先不用：各說一個理由。
# 答：
#
# (2) cabin 缺 77%、age 缺 20%。
#     (a) 讀上面的輸出：age 缺值者與有年齡者的生還率各是多少？「缺值」本身是不是隨機的？
#     (b) 本章用「中位數補值」處理 age。對一棵決策樹來說，補成中位數的那一群人會在 age 軸上擠成一根柱子；
#         樹有沒有辦法把這群人單獨切出來？（提示：一個值的左右各切一刀）
#     (c) cabin 先丟掉。有沒有比「整欄丟掉」更好的用法？（提示：有沒有 cabin 本身是不是一個訊號？）
# 答：
#
# (3) 先預測：如果一棵樹只能問「一個是非題」，它會問什麼？只問這一題，accuracy 大約多少？
#     （用上面的生還率與人數手算：每一組都猜「多數」，猜對的人數加總 ÷ 總人數）
# 答：
#
# (4) 先預測：第三、四章都花了很多力氣討論「要不要標準化」。決策樹需要標準化嗎？
#     fare 這麼右偏，要不要先取 log？（先寫下你的預測，3D 的實驗 3 會驗證）
# 答：

---
## 2. 前處理與 Baseline

#### 前處理（整章共用）

決策樹只能吃數字矩陣，所以先把 7 個特徵變成 $d = 9$ 個數值欄位；前處理一律包進 `ColumnTransformer`，只用訓練集 fit：

| 原始欄位 | 處理 | 變成 |
| --- | --- | --- |
| `pclass`、`age`、`sibsp`、`parch`、`fare` | 缺值補**中位數** | 原名 5 欄 |
| `sex` | `OrdinalEncoder`：female → 0、male → 1 | `sex` 1 欄 |
| `embarked` | 缺值補眾數，再 one-hot | `embarked_C`、`embarked_Q`、`embarked_S` 3 欄 |

`sex` 只有兩個值，0／1 一欄就夠；`embarked` 有三個值，如果編成 0／1／2，樹只能切出「C | Q, S」或「C, Q | S」兩種分法（被假的順序綁住），one-hot 則每個港口都能單獨切出來。

#### Baseline

- **Dummy prior**：永遠輸出訓練集的生還比例。
- **性別規則**：女性 → 生還、男性 → 罹難。這是「只問一個問題」的人工規則，也就是一棵**深度 1 的樹**（decision stump）可能學到的東西。

#### 本節名詞

- **accuracy**：猜對的比例。
- **log loss**：$\frac{1}{n}\sum_i -\ln P(\text{第 } i \text{ 筆的正確類別})$；Dummy prior 的 log loss = 標籤分布的熵（第三章 TODO 2）。
- **5-fold 分層 CV**（`StratifiedKFold`）：訓練集切 5 份，輪流 1 份驗證、其餘訓練；每份的生還比例都跟全體一樣。報告「平均 ± 標準差」。

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder

NUM = ["pclass", "age", "sibsp", "parch", "fare"]


def make_prep() -> ColumnTransformer:
    """回傳一個新的（未 fit）前處理器：pd.DataFrame（原始 7 欄）→ pd.DataFrame（9 個數值欄）。"""
    return ColumnTransformer(
        [
            ("num", SimpleImputer(strategy="median"), NUM),
            ("sex", OrdinalEncoder(), ["sex"]),
            (
                "emb",
                Pipeline(
                    [
                        ("imp", SimpleImputer(strategy="most_frequent")),
                        ("oh", OneHotEncoder(sparse_output=False)),
                    ]
                ),
                ["embarked"],
            ),
        ],
        verbose_feature_names_out=False,
    ).set_output(transform="pandas")


prep = make_prep().fit(X_train)
Z_train = prep.transform(X_train)  # (N, 9) 的 DataFrame
Z_test = prep.transform(X_test)
FEATURES = Z_train.columns.tolist()
# 手寫模型用的 NumPy 版本（3、4 節共用）
Xtr = Z_train.to_numpy(dtype=float)  # (N, d)
Xte = Z_test.to_numpy(dtype=float)
ytr = y_train.to_numpy()  # (N,) 0／1
yte = y_test.to_numpy()
print("特徵：", FEATURES)
print("sex 的編碼：", dict(zip(prep.named_transformers_["sex"].categories_[0], [0, 1])))
display(Z_train.head())

In [ ]:
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score, cross_validate

# 整章共用的 5-fold（分層、打亂、固定 seed）
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)


class SexRule(ClassifierMixin, BaseEstimator):
    """人工規則：女性 → 1（生還）、男性 → 0。輸入是前處理後的矩陣（要有 sex 欄）。"""

    def fit(self, X: pd.DataFrame, y: pd.Series) -> "SexRule":
        self.classes_ = np.array([0, 1])
        return self

    def predict(self, X: pd.DataFrame) -> np.ndarray:
        return (X["sex"].to_numpy() == 0).astype(int)


def cv_summary(model: BaseEstimator, X: pd.DataFrame = Z_train, y: pd.Series = y_train) -> dict[str, str]:
    """5-fold CV。回傳 {"accuracy": "平均 ± 標準差", "log_loss": "平均 ± 標準差"}；沒有 predict_proba 的模型 log_loss 為 "—"。"""
    scoring = {"accuracy": "accuracy"}
    if hasattr(model, "predict_proba"):
        scoring["log_loss"] = "neg_log_loss"
    s = cross_validate(model, X, y, cv=cv, scoring=scoring)
    out = {"accuracy": f"{s['test_accuracy'].mean():.4f} ± {s['test_accuracy'].std():.4f}", "log_loss": "—"}
    if "log_loss" in scoring:
        v = -s["test_log_loss"]
        out["log_loss"] = f"{v.mean():.4f} ± {v.std():.4f}"
    return out


baseline = pd.DataFrame(
    {
        "Dummy most_frequent": cv_summary(DummyClassifier(strategy="most_frequent")),
        "Dummy prior": cv_summary(DummyClassifier(strategy="prior")),
        "Sex rule": cv_summary(SexRule()),
    }
).T
display(baseline)

In [ ]:
# TODO 2（問答）
#
# (1) Dummy prior 的 log loss 用生還比例手算出來（自然對數）。
#     (a) 寫出公式並代入 train 的生還比例 q。
#     (b) 跟第三章二元的 0.66、上限 ln 2 ≈ 0.693 比，這份資料比較平衡還是比較不平衡？
# 答：
#
# (2) 性別規則只看一個欄位就比 Dummy 高十幾個百分點。
#     (a) 用 EDA 的數字解釋：這條規則在女性、男性兩組各猜對多少比例？加權平均起來是多少？跟 CV 結果一致嗎？
#     (b) 這條規則能不能被「一棵深度 1 的樹」學出來？樹會不會選別的欄位當第一刀？（3B 會驗證）
#     (c) 之後的樹如果 CV accuracy 只比這條規則高一點點，你會怎麼解讀？
# 答：

---
## 3. 手寫模型

整棵樹是由一個小動作重複組成的：**在一個節點上，從所有特徵、所有切點裡，找出讓子節點「最純」的那一刀**。這一節由下往上寫：

| 小節 | 做什麼 | TODO |
| --- | --- | --- |
| 3A | 不純度：一個節點有多「混」 | 3 |
| 3B | 在**一個特徵**上一次算完所有切點的不純度下降量，挑最好的 | 4、5、6 |
| 3C | 把 3B 遞迴地用下去，長成一棵樹；預測；特徵重要度 | 8、9、10 |
| 3D | 實驗：不純度的種類、深度、單調轉換、不穩定性 | — |

### 3A. 不純度（impurity）

一個節點裡的樣本如果全是同一類，就叫「純」（pure），預測時毫無疑問；兩類各半，就是最「混」的狀態。不純度把「有多混」變成一個數字：

| 名稱 | 公式（$p_k$ = 節點裡第 $k$ 類的比例） | 二元、$p = (0.5, 0.5)$ | 純的節點 |
| --- | --- | --- | --- |
| Gini | $G = \sum_k p_k (1 - p_k) = 1 - \sum_k p_k^2$ | 0.5 | 0 |
| entropy（以 2 為底） | $H = -\sum_k p_k \log_2 p_k$（$0 \log 0 = 0$） | 1 | 0 |
| 誤分類率（misclassification） | $E = 1 - \max_k p_k$ | 0.5 | 0 |

sklearn 預設用 Gini（`criterion="gini"`），也可以選 `"entropy"`／`"log_loss"`（兩者在 sklearn 裡是同一個東西）。誤分類率 sklearn 沒有提供，3D 的實驗 1 會看為什麼。

#### 符號表（3A）

| 符號 | 意義 | shape | 程式裡 |
| --- | --- | --- | --- |
| counts | 每個節點（或每個候選子節點）各類別的**筆數**；一列是一個節點 | $(m, K)$ | `counts` |
| $n$ | 每一列的總筆數 | $(m, 1)$ | `n = counts.sum(axis=-1, keepdims=True)` |
| $p$ | 比例 $= \text{counts} / n$；$n = 0$ 的列定義為全 0 | $(m, K)$ | `p` |
| 回傳 | 每一列的不純度 | $(m,)$ | — |

函數吃的是**筆數**不是比例，而且一次處理很多列：3B 會一次把某個特徵的所有切點的左、右子節點筆數排成 $(n-1, K)$ 的矩陣丟進來。也要能吃一維的 $(K,)$（單一節點），所以加總用 `axis=-1`（最後一軸）。

**下標公式**（TODO 3）：$p_{ik} = c_{ik} / \sum_l c_{il}$，$G_i = \sum_k p_{ik}(1 - p_{ik})$ —— 保留 $i$（哪一列）、加總 $k$（類別）→ `.sum(axis=-1)`。

#### 最小例子：TODO 3（Gini 與 entropy）

3 個節點：`counts = [[5, 5], [9, 1], [10, 0]]`。

1. **正確答案**：一列一列、一個類別一個類別用 Python 迴圈算。目標：Gini `[0.5, 0.18, 0]`、entropy `[1, 0.469, 0]`。
2. **積木**：`n` 用 `keepdims=True` 才能跟 `(3, 2)` 廣播；`np.divide(..., where=...)` 處理 $n = 0$（空節點的 $p$ 定義為全 0）；`np.log2(0)` 會變成 `-inf`、`0 * -inf` 是 `nan`。
3. **留給你**：用 `p` 組出兩個公式，並處理 $0 \log 0$。

In [ ]:
# 最小例子：TODO 3（Gini 與 entropy）
import math

counts_mini = np.array([[5, 5], [9, 1], [10, 0]], dtype=float)  # (m=3, K=2)

# ---- 1. 正確答案：迴圈 ----
for row_mini in counts_mini:
    n_row = row_mini.sum()
    g_loop = 1 - sum((c / n_row) ** 2 for c in row_mini)
    h_loop = -sum((c / n_row) * math.log2(c / n_row) for c in row_mini if c > 0) + 0.0  # c = 0 那一項直接跳過；+ 0.0 把 -0.0 變成 0.0
    print(f"counts={row_mini}  Gini={g_loop:.4f}  entropy={h_loop:.4f}")

# ---- 2. 積木 ----
n_mini = counts_mini.sum(axis=-1, keepdims=True)
print("\nn = counts.sum(axis=-1, keepdims=True)", n_mini.shape, n_mini.ravel(), "← (3, 1) 才能跟 (3, 2) 廣播")
print("counts.sum(axis=-1)（沒有 keepdims）  ", counts_mini.sum(axis=-1).shape, "← (3,) 跟 (3, 2) 會對不齊")
p_mini = np.divide(counts_mini, n_mini, out=np.zeros_like(counts_mini), where=n_mini > 0)
print("p = counts / n                        ", p_mini.shape, "\n", p_mini)
print("p ** 2 的每一列加總                     ", (p_mini**2).sum(axis=-1))
print("p * (1 − p) 的每一列加總                ", (p_mini * (1 - p_mini)).sum(axis=-1), "← 跟 1 − Σp² 一樣（因為每列 Σp = 1）")
with np.errstate(divide="ignore", invalid="ignore"):
    print("np.log2(p)                            \n", np.log2(p_mini), "← log2(0) = -inf")
    print("p * np.log2(p)                        \n", p_mini * np.log2(p_mini), "← 0 * -inf = nan，要換成 0")
print("空節點 counts=[0, 0]：p =", np.divide(np.zeros(2), 0.0, out=np.zeros(2), where=False),
      "→ 1 − Σp² =", 1 - 0.0, "（✗ 空節點不該是 1）、Σp(1−p) =", 0.0)

# ---- 3. 留給你 ----
# 問題：Gini 用哪一個積木（想想空節點）？entropy 要怎麼用 np.where 把 p = 0 的那幾項換成 0，再沿哪一軸加總、加什麼號？
# 寫完 TODO、執行過下一格之後，取消下面的註解來對照：
# print(gini(counts_mini), entropy(counts_mini))

In [ ]:
from numpy.typing import ArrayLike


def _proportions(counts: ArrayLike) -> np.ndarray:
    """counts (..., K) → p (..., K)，每一列除以列總和；總和為 0 的列給全 0。"""
    counts = np.asarray(counts, dtype=float)
    n = counts.sum(axis=-1, keepdims=True)
    return np.divide(counts, n, out=np.zeros_like(counts), where=n > 0)


def gini(counts: ArrayLike) -> np.ndarray:
    """Gini 不純度。counts (..., K) → (...,)：Σ_k p_k (1 − p_k)（非空節點 = 1 − Σ_k p_k²）。"""
    p = _proportions(counts)
    # TODO 3（程式）之一：Gini
    #   - p 的 shape 是 (..., K)，類別在最後一軸 → 對 axis=-1 加總
    #   - 常見錯誤：寫成 1 − Σ p²。非空節點兩者相等（因為 Σ p = 1），
    #     但空節點的 p 是全 0（_proportions 的約定），1 − Σ p² 會變成 1 → 用 Σ p (1 − p) 才會是 0
    raise NotImplementedError("TODO 3：gini")


def entropy(counts: ArrayLike) -> np.ndarray:
    """Entropy（以 2 為底）。counts (..., K) → (...,)：−Σ_k p_k log₂ p_k，0·log 0 = 0。"""
    p = _proportions(counts)
    # TODO 3（程式）之二：entropy
    #   - p = 0 的位置 np.log2 會給 -inf、乘起來是 nan → 用 np.where(p > 0, p * np.log2(p), 0.0) 換成 0
    #   - 用 with np.errstate(divide="ignore", invalid="ignore"): 包住，避免警告
    #   - 別忘了前面的負號
    raise NotImplementedError("TODO 3：entropy")


def misclassification(counts: ArrayLike) -> np.ndarray:
    """誤分類率。counts (..., K) → (...,)：1 − max_k p_k（3D 實驗 1 用）。"""
    p = _proportions(counts)
    return 1 - p.max(axis=-1)


from scipy.stats import entropy as scipy_entropy

test_counts = np.array([[5, 5], [9, 1], [10, 0], [3, 3], [0, 7], [1, 2]], dtype=float)
print("✅ Gini：", gini(test_counts).round(4), "（應為 [0.5 0.18 0. 0.5 0. 0.4444]）")
print(
    "✅ entropy 與 scipy 的最大差異：",
    np.abs(entropy(test_counts) - scipy_entropy(test_counts.T, base=2)).max(),
    "（應 < 1e-12）",
)
print("✅ 單一節點（一維輸入）：gini([400, 400]) =", gini([400, 400]), "（應 = 0.5）")
print("✅ 空節點：gini([0, 0]) == 0：", bool(gini([0, 0]) == 0), "、entropy([0, 0]) == 0：", bool(entropy([0, 0]) == 0), "（都應 True；不能是 1 或 nan）")

In [ ]:
# 二元時三種不純度的形狀：x 軸是生還比例 p
p_grid = np.linspace(0, 1, 201)
c_grid = np.stack([1 - p_grid, p_grid], axis=1) * 100  # 假裝每個節點 100 筆
fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(p_grid, gini(c_grid), label="Gini")
ax.plot(p_grid, entropy(c_grid), label="entropy (bits)")
ax.plot(p_grid, entropy(c_grid) / 2, "--", label="entropy / 2")
ax.plot(p_grid, misclassification(c_grid), label="misclassification")
ax.set_xlabel("p = proportion of class 1 in the node")
ax.set_ylabel("impurity")
ax.legend()
ax.set_title("Impurity measures (binary)")
plt.show()

### 3B. 在一個特徵上找最好的切點

固定一個節點、一個特徵 $j$。把節點裡的 $n$ 筆樣本依 $x_j$ 由小到大排好：$x_{(0)} \le x_{(1)} \le \dots \le x_{(n-1)}$。
「切在第 $i$ 個和第 $i+1$ 個之間」= 前 $i+1$ 筆去左邊、其餘去右邊，$i = 0, \dots, n-2$，共 $n - 1$ 個候選切點。

對每一個候選切點，算**不純度下降量**（impurity decrease）：

$$\Delta I = I(t) - \left[\frac{n_L}{n} I(t_L) + \frac{n_R}{n} I(t_R)\right]$$

- 中括號裡是子節點不純度的**加權平均**（前置複習 4），權重是人數比例。
- $\Delta I$ 越大，切完越純。選 $\Delta I$ 最大的切點。
- 用 Gini 時 $\Delta I$ 叫 Gini decrease；用 entropy 時叫**資訊增益**（information gain）。

**有效率的關鍵：累加。** 天真的做法是每個切點都重新數一次左邊有幾個生還、幾個罹難（每個切點 $O(n)$，共 $O(n^2)$）。
排好序之後，「切點往右移一格」只是把一筆樣本從右邊搬到左邊 → 左邊的類別筆數就是 one-hot 的**累加和**（前置複習 13），一次 `cumsum` 就得到全部 $n - 1$ 個切點的左邊筆數，右邊 = 總數 − 左邊。整個特徵 $O(n \log n)$（排序最貴）。

**兩個限制**：

1. **相同的值不能拆開**：如果 $x_{(i)} = x_{(i+1)}$，「切在它們之間」沒有任何門檻做得到（門檻只能說「$\le s$」）→ 這個位置無效。只有 $x_{(i)} < x_{(i+1)}$ 的位置能切。
2. **葉子最少樣本數** `min_samples_leaf`：左右兩邊都至少要有這麼多筆。

**門檻取中點**：有效位置 $i$ 的門檻 $s = \frac{x_{(i)} + x_{(i+1)}}{2}$（sklearn 也是），左邊 $= \{x_j \le s\}$。

#### 符號表（3B）

| 符號 | 意義 | shape | 程式裡 |
| --- | --- | --- | --- |
| $x$ | 這個節點所有樣本的第 $j$ 個特徵 | $(n,)$ | `x` |
| $y$ | 這些樣本的類別**編號**（0..K−1） | $(n,)$ | `y_idx` |
| order | 讓 $x$ 由小到大的位置 | $(n,)$ | `order` |
| $x_{(i)}$、$y_{(i)}$ | 排序後的第 $i$ 個 | $(n,)$ | `xs`、`ys` |
| onehot | `ys` 的 one-hot | $(n, K)$ | `onehot` |
| left | 切在第 $i$ 個之後時，左邊各類的筆數 | $(n-1, K)$ | `left` |
| total | 整個節點各類的筆數 | $(K,)$ | `total` |
| right | 右邊各類的筆數 = total − left | $(n-1, K)$ | `right` |
| $n_L$、$n_R$ | 左、右的總筆數：$n_L = i + 1$、$n_R = n - n_L$ | $(n-1,)$ | `n_left`、`n_right` |
| $\Delta I$ | 每個切點的不純度下降量 | $(n-1,)$ | `gain` |
| valid | 這個切點能不能用（值不同、兩邊人數夠） | $(n-1,)$ bool | `valid` |

**下標公式**：

- TODO 4：$\text{left}_{ik} = \sum_{r=0}^{i} \text{onehot}_{rk}$ —— 保留 $i$、$k$，對 $r \le i$ 累加 → `np.cumsum(onehot, axis=0)`，再去掉最後一列（「全部去左邊」不是切點）。
- TODO 5：$\Delta I_i = I(\text{total}) - \frac{n_{L,i}}{n} I(\text{left}_i) - \frac{n_{R,i}}{n} I(\text{right}_i)$ —— 每個 $i$ 各自算，沒有加總。
- TODO 6：$\text{valid}_i = [x_{(i)} < x_{(i+1)}] \wedge [n_{L,i} \ge m] \wedge [n_{R,i} \ge m]$（$m$ = `min_samples_leaf`）；門檻 $s_i = (x_{(i)} + x_{(i+1)})/2$。

**$n = 4$ 的手算例**：$x = (3, 1, 2, 2)$、$y = (1, 0, 0, 1)$。排序後 $x_s = (1, 2, 2, 3)$、$y_s = (0, 0, 1, 1)$。

| 切在第 $i$ 個之後 | 左邊 | 右邊 | left | valid | 門檻 |
| --- | --- | --- | --- | --- | --- |
| $i = 0$ | $\{1\}$ | $\{2, 2, 3\}$ | (1, 0) | ✓（1 < 2） | 1.5 |
| $i = 1$ | $\{1, 2\}$ | $\{2, 3\}$ | (2, 0) | ✗（2 = 2，兩個 2 不能拆開） | — |
| $i = 2$ | $\{1, 2, 2\}$ | $\{3\}$ | (2, 1) | ✓（2 < 3） | 2.5 |

#### 最小例子：TODO 4（累加出每個切點的左邊筆數）

就用上面 $n = 4$ 的手算例。

1. **正確答案**：對每個切點 $i$，用迴圈數「排序後前 $i+1$ 筆」裡各類有幾個。目標 `left = [[1, 0], [2, 0], [2, 1]]`。
2. **積木**：`argsort` 一起排 `x`、`y`；`np.eye(K)[ys]` 的 one-hot；`np.cumsum` 有沒有 `axis` 的差別。
3. **留給你**：哪一個 `cumsum`、要不要去掉最後一列？

In [ ]:
# 最小例子：TODO 4（累加出每個切點的左邊筆數）
x_mini = np.array([3.0, 1.0, 2.0, 2.0])
y_mini = np.array([1, 0, 0, 1])
K_mini = 2

# ---- 1. 正確答案：迴圈 ----
order_mini = np.argsort(x_mini, kind="stable")
xs_mini, ys_mini = x_mini[order_mini], y_mini[order_mini]
left_loop = []
for i_mini in range(len(xs_mini) - 1):
    left_loop.append([int((ys_mini[: i_mini + 1] == k).sum()) for k in range(K_mini)])
print("排序後 xs =", xs_mini, " ys =", ys_mini)
print("正確答案 left =", left_loop)

# ---- 2. 積木 ----
onehot_mini = np.eye(K_mini)[ys_mini]
print("\nonehot = np.eye(K)[ys]  ", onehot_mini.shape, "\n", onehot_mini)
print("np.cumsum(onehot)（沒給 axis）", np.cumsum(onehot_mini), "← 攤平成一維再累加，類別混在一起了")
print("np.cumsum(onehot, axis=1)", "\n", np.cumsum(onehot_mini, axis=1), "← 沿著「類別」累加，沒有意義")
print("total = onehot.sum(axis=0)", onehot_mini.sum(axis=0))

# ---- 3. 留給你 ----
# 問題：沿著哪一軸累加，第 i 列才會是「前 i+1 筆」各類的筆數？累加完是 (4, 2)，但切點只有 3 個 —— 哪一列要去掉、為什麼？
# 寫完 TODO 4、5、6，執行過定義 class DecisionTree 的那一格之後，取消下面的註解來對照：
# print(DecisionTree()._split_stats(np.array([3.0, 1.0, 2.0, 2.0]), np.array([1, 0, 0, 1]), 2)["left"])

#### 最小例子：TODO 5（每個切點的不純度下降量）

接著上一格：`left = [[1, 0], [2, 0], [2, 1]]`、`total = [2, 2]`，用 Gini。

1. **正確答案**：一個切點一個切點用公式算。目標 $\Delta I = (0.1667, 0.5, 0.1667)$。
2. **積木**：`right = total - left` 的廣播、`gini(left)` 一次算三列、`n_left`／`n_right` 的值。
3. **留給你**：把三個積木組成 $\Delta I$。

In [ ]:
# 最小例子：TODO 5（每個切點的不純度下降量）
left_mini = np.array([[1, 0], [2, 0], [2, 1]], dtype=float)  # (n-1=3, K=2)
total_mini = np.array([2, 2], dtype=float)  # (K,)
n_all_mini = 4

# ---- 1. 正確答案：迴圈 ----
for i_mini, l_row in enumerate(left_mini):
    r_row = total_mini - l_row
    nl, nr = l_row.sum(), r_row.sum()
    g_l = 1 - sum((c / nl) ** 2 for c in l_row)
    g_r = 1 - sum((c / nr) ** 2 for c in r_row)
    g_t = 1 - sum((c / n_all_mini) ** 2 for c in total_mini)
    print(f"i={i_mini}  left={l_row} right={r_row}  ΔI = {g_t:.4f} − ({nl:.0f}/4·{g_l:.4f} + {nr:.0f}/4·{g_r:.4f}) = {g_t - nl / 4 * g_l - nr / 4 * g_r:.4f}")

# ---- 2. 積木 ----
right_mini = total_mini - left_mini
print("\nright = total − left   ", right_mini.shape, "\n", right_mini, "← (K,) 廣播到 (3, K)")
n_left_mini = np.arange(1, n_all_mini)
print("n_left = arange(1, n)  ", n_left_mini, "；n_right = n − n_left =", n_all_mini - n_left_mini)
print("gini(left)             ", gini(left_mini).round(4))
print("gini(right)            ", gini(right_mini).round(4))
print("gini(total)            ", gini(total_mini), "← 純量，會廣播")

# ---- 3. 留給你 ----
# 問題：ΔI = gini(total) − (哪個 / n) * gini(left) − (哪個 / n) * gini(right)？注意 n_left 的 shape 是 (3,)，gini(left) 也是 (3,)。
# 注意：i = 1 的 ΔI 最大（0.5），但 xs[1] = xs[2] = 2，這個切點其實做不到 —— TODO 6 會把它排除。

#### 最小例子：TODO 6（有效的切點與門檻）

接著上一格：`xs = [1, 2, 2, 3]`、`gain = [0.1667, 0.5, 0.1667]`，`min_samples_leaf = 1`。

1. **正確答案**：迴圈檢查每個位置。目標 `valid = [True, False, True]`、最好的有效切點 $i = 0$（平手時取第一個）、門檻 1.5。
2. **積木**：`xs[:-1]`、`xs[1:]` 兩個錯開一格的切片；`np.where(valid, gain, -np.inf)`；`np.argmax` 平手取第一個。
3. **留給你**：組出 `valid` 與門檻。

In [ ]:
# 最小例子：TODO 6（有效的切點與門檻）
xs_mini = np.array([1.0, 2.0, 2.0, 3.0])
gain_mini = np.array([1 / 6, 0.5, 1 / 6])
n_left_mini = np.array([1, 2, 3])
n_right_mini = 4 - n_left_mini
msl_mini = 1

# ---- 1. 正確答案：迴圈 ----
valid_loop = [
    bool(xs_mini[i] < xs_mini[i + 1] and n_left_mini[i] >= msl_mini and n_right_mini[i] >= msl_mini)
    for i in range(3)
]
best_loop = max((i for i in range(3) if valid_loop[i]), key=lambda i: gain_mini[i])  # max 平手時取第一個
print("正確答案 valid =", valid_loop, " 最好的 i =", best_loop, " 門檻 =", (xs_mini[best_loop] + xs_mini[best_loop + 1]) / 2)

# ---- 2. 積木 ----
print("\nxs[:-1] =", xs_mini[:-1], "← 第 i 個")
print("xs[1:]  =", xs_mini[1:], "← 第 i+1 個；兩者逐項比較 = 每個切點左右的兩個值")
print("np.where(valid, gain, -inf) 搭配 argmax：被排除的位置永遠不會被選上")
print("np.argmax([0.2, 0.5, 0.5]) =", np.argmax([0.2, 0.5, 0.5]), "← 平手取第一個（門檻最小的那個）")

# ---- 3. 留給你 ----
# 問題：valid 由哪三個條件 & 起來？門檻是 xs 的哪兩個切片的平均？
# 寫完 TODO、執行過下一格之後，取消下面的註解來對照：
# print(DecisionTree().best_split_on_feature(np.array([3.0, 1.0, 2.0, 2.0]), np.array([1, 0, 0, 1]), 2))

#### 程式結構圖：`DecisionTree` 由哪些方法組成

這一格先寫 3B 的兩個方法（找一個特徵上的最好切點），3C 再用**繼承**把長樹、預測補上去（`class DecisionTree(DecisionTree)` 的寫法只是為了把一個 class 拆成好幾格，最後得到的是同一個 class）。

```
DecisionTree(criterion="gini", max_depth=None, min_samples_split=2, min_samples_leaf=1)
├── 超參數：criterion、max_depth、min_samples_split、min_samples_leaf
│
├── _impurity(counts)                      依 criterion 呼叫 gini() 或 entropy()（3A）
├── _split_stats(x, y_idx, K)              一個特徵、所有切點的統計量         ← TODO 4、5
│     → dict：xs、left、right、n_left、n_right、gain
├── best_split_on_feature(x, y_idx, K)     → (最好的 ΔI, 門檻)                ← TODO 6
├── best_split(X, y_idx)                   走過 d 個特徵，各呼叫一次上面那個 → (ΔI, j, s)
│
│   ---- 3C ----
├── fit(X, y)          存下 X_、classes_、y_idx_，呼叫 _grow(全部的列號, 0)，結果存在 root_
├── _grow(idx, depth)  遞迴：建一個 Node；能切就切，左右各再 _grow 一次      ← TODO 8
├── apply_one(x)       一筆樣本從 root_ 一路走到葉子，回傳那個 Node         ← TODO 9
├── predict_proba(X) / predict(X)
└── feature_importances_                                                     ← TODO 10
```

這一節所有方法的 `x`、`y_idx` 都只是**某一個節點**裡的樣本（長樹時會一直傳子集進來），不是整份訓練資料。

In [ ]:
from typing import Literal, Self


class DecisionTree:
    """CART 分類樹（只處理數值特徵、二元切分）。結構仿 sklearn：fit / predict / predict_proba。"""

    def __init__(
        self,
        criterion: Literal["gini", "entropy"] = "gini",
        max_depth: int | None = None,
        min_samples_split: int = 2,
        min_samples_leaf: int = 1,
    ) -> None:
        self.criterion = criterion
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.min_samples_leaf = min_samples_leaf

    def _impurity(self, counts: ArrayLike) -> np.ndarray:
        """counts (..., K) → (...,)：依 criterion 算不純度。"""
        return gini(counts) if self.criterion == "gini" else entropy(counts)

    def _split_stats(self, x: np.ndarray, y_idx: np.ndarray, K: int) -> dict[str, np.ndarray]:
        """一個特徵上「所有」切點的統計量。x、y_idx (n,) → dict，切點相關的陣列都是 (n−1,) 或 (n−1, K)。

        切點 i = 「排序後前 i+1 筆去左邊」，i = 0..n−2。
        """
        n = len(x)
        order = np.argsort(x, kind="stable")
        xs, ys = x[order], y_idx[order]
        onehot = np.eye(K)[ys]  # (n, K)
        total = onehot.sum(axis=0)  # (K,)
        # TODO 4（程式）：每個切點的左邊各類筆數 left (n−1, K)
        #   - 下標公式：left[i, k] = Σ_{r ≤ i} onehot[r, k] → 沿著「樣本」那一軸累加
        #   - 累加完是 (n, K)：最後一列 = 全部去左邊、右邊沒人，不是切點 → 去掉
        raise NotImplementedError("TODO 4：left")
        right = total - left  # (n−1, K)
        n_left = np.arange(1, n)  # (n−1,)：切點 i 的左邊有 i+1 筆
        n_right = n - n_left
        # TODO 5（程式）：每個切點的不純度下降量 gain (n−1,)
        #   - ΔI = I(total) − [n_L/n · I(left) + n_R/n · I(right)]
        #   - 用 self._impurity(...)，它可以一次吃 (n−1, K) 的整個矩陣
        raise NotImplementedError("TODO 5：gain")
        return {"xs": xs, "left": left, "right": right, "n_left": n_left, "n_right": n_right, "gain": gain}

    def best_split_on_feature(self, x: np.ndarray, y_idx: np.ndarray, K: int) -> tuple[float, float]:
        """一個特徵上最好的有效切點 → (ΔI, 門檻)；沒有任何有效切點時回傳 (-inf, nan)。"""
        if len(x) < 2:
            return -np.inf, np.nan
        s = self._split_stats(x, y_idx, K)
        xs, gain = s["xs"], s["gain"]
        # TODO 6（程式）：有效的切點與門檻
        #   - valid (n−1,)：xs[i] < xs[i+1]（相同的值不能拆開），且 n_left、n_right 都 ≥ self.min_samples_leaf
        #   - 用 xs[:-1]、xs[1:] 兩個錯開一格的切片逐項比較
        #   - 沒有任何 valid 就回傳 (-np.inf, np.nan)
        #   - i = np.argmax(np.where(valid, gain, -np.inf))；門檻取 xs[i]、xs[i+1] 的中點
        raise NotImplementedError("TODO 6：valid、門檻")

    def best_split(self, X: np.ndarray, y_idx: np.ndarray) -> tuple[float, int, float]:
        """在一個節點上找最好的 (ΔI, 特徵 j, 門檻 s)；X (n, d)。沒有任何有效切點時 j = −1。

        特徵依 0..d−1 的順序比較，ΔI 要「嚴格大於」目前最好的才換 → 平手時留下編號小的特徵。
        """
        best = (-np.inf, -1, np.nan)
        for j in range(X.shape[1]):
            g, s = self.best_split_on_feature(X[:, j], y_idx, self.n_classes_)
            if g > best[0] + 1e-12:  # 浮點誤差內的差異視為平手
                best = (g, j, s)
        return best

In [ ]:
# 檢查點：用最小例子與一個「暴力法」對照
dt_check = DecisionTree()
s_mini = dt_check._split_stats(np.array([3.0, 1.0, 2.0, 2.0]), np.array([1, 0, 0, 1]), 2)
print("✅ left：", s_mini["left"].tolist(), "（應為 [[1, 0], [2, 0], [2, 1]]）")
print("✅ gain：", s_mini["gain"].round(4), "（應為 [0.1667 0.5 0.1667]）")
print(
    "✅ best_split_on_feature：",
    dt_check.best_split_on_feature(np.array([3.0, 1.0, 2.0, 2.0]), np.array([1, 0, 0, 1]), 2),
    "（應為 (0.1666..., 1.5)）",
)


def brute_force_split(x: np.ndarray, y_idx: np.ndarray, K: int, crit=gini) -> tuple[float, float]:
    """暴力法：每個候選門檻都重新數一次左右的筆數（O(n²)，只用來對照）。"""
    vals = np.unique(x)
    best = (-np.inf, np.nan)
    for a, b in zip(vals[:-1], vals[1:]):
        s = (a + b) / 2
        go = x <= s
        cl, cr = np.bincount(y_idx[go], minlength=K), np.bincount(y_idx[~go], minlength=K)
        g = crit(cl + cr) - (go.sum() * crit(cl) + (~go).sum() * crit(cr)) / len(x)
        if g > best[0] + 1e-12:
            best = (float(g), float(s))
    return best


dt_check.n_classes_ = 2
for j, name in enumerate(FEATURES):
    mine = dt_check.best_split_on_feature(Xtr[:, j], ytr, 2)
    ref = brute_force_split(Xtr[:, j], ytr, 2)
    ok = np.isclose(mine[0], ref[0]) and np.isclose(mine[1], ref[1])
    print(f"{'✅' if ok else '❌'} {name:<11} ΔI = {mine[0]:.4f}  門檻 = {mine[1]:<8.4g} 暴力法 ({ref[0]:.4f}, {ref[1]:.4g})")

In [ ]:
# 根節點：每個特徵「最好的那一刀」能讓 Gini 下降多少？
# 左圖：每個特徵最好的 ΔI；右圖：age 與 fare 在所有切點上的 ΔI 曲線
root_gain = pd.Series(
    {name: dt_check.best_split_on_feature(Xtr[:, j], ytr, 2)[0] for j, name in enumerate(FEATURES)}
).sort_values()

fig, axes = plt.subplots(1, 2, figsize=(13, 3.8))
axes[0].barh(root_gain.index, root_gain.values, color="#4C72B0")
axes[0].set_title("best Gini decrease at the root, per feature")
for name, ax_c in [("age", "C0"), ("fare", "C1")]:
    j = FEATURES.index(name)
    s = dt_check._split_stats(Xtr[:, j], ytr, 2)
    valid = s["xs"][:-1] < s["xs"][1:]
    mids = (s["xs"][:-1] + s["xs"][1:]) / 2
    axes[1].plot(mids[valid], s["gain"][valid], ".", ms=3, color=ax_c, label=name)
axes[1].set_xscale("symlog")
axes[1].set_xlabel("threshold (symlog scale)")
axes[1].set_ylabel("Gini decrease")
axes[1].legend()
axes[1].set_title("Gini decrease over all thresholds")
fig.tight_layout()
plt.show()

root_counts = np.bincount(ytr, minlength=2)
print("根節點 counts（罹難, 生還）：", root_counts, " Gini =", round(float(gini(root_counts)), 4))
for sex_val, sex_name in [(0, "female"), (1, "male")]:
    c = np.bincount(ytr[Xtr[:, FEATURES.index("sex")] == sex_val], minlength=2)
    print(f"  {sex_name:<6} counts：{c}  Gini = {float(gini(c)):.4f}")

In [ ]:
# TODO 7（問答）根節點的第一刀
#
# (1) 用上一格印出的 counts 手算（寫出算式）：
#     (a) 根節點的 Gini。
#     (b) 依 sex 切開後，左（female）、右（male）兩個子節點的 Gini，以及它們的加權平均。
#     (c) ΔI = (a) − (b)。跟左圖 sex 那一條一致嗎？
# 答：
#
# (2) 第一刀選了 sex，和你在 TODO 1 (3) 的預測一樣嗎？左圖第二名是哪個特徵、差多少？
#     sex 只有 0／1 兩個值，候選切點只有幾個？門檻為什麼是 0.5？
# 答：
#
# (3) 看右圖 age 的 ΔI 曲線：
#     (a) 曲線在哪一段比較高？對照 EDA 的年齡分組生還率，最好的 age 門檻落在哪裡、代表什麼樣的人？
#     (b) 曲線是鋸齒狀的，而且很多門檻的 ΔI 都很接近最高點。如果訓練資料換一批（多幾個、少幾個人），
#         「最好的門檻」會很穩定嗎？（3D 的實驗 4 會回來看）
# 答：
#
# (4) _split_stats 用排序 + cumsum，成本是 O(n log n)；暴力法對每個門檻重新數一次是 O(n²)。
#     (a) 一個節點要對 d 個特徵各做一次；一整層的節點加起來大約處理多少筆資料？
#     (b) 樹深度 D 的話，整棵樹大約是 O(?)。（提示：同一層的節點各自拿到的樣本互不重疊，加總 = N）
# 答：

### 3C. 長成一棵樹：遞迴、預測、特徵重要度

**長樹（`_grow`）**：從根節點（全部訓練資料）開始，

1. 建一個 `Node`，記下這個節點的類別筆數 `value`、樣本數、不純度；
2. 檢查**終止條件**，成立就直接回傳（它就是葉子）：
   - 已經是純的（不純度 = 0）
   - 樣本數 < `min_samples_split`
   - 深度已達 `max_depth`
3. 否則用 `best_split` 找最好的 $(j, s)$；找不到有效切點也當葉子；
4. 記下 $(j, s)$，把樣本依 $x_j \le s$ 分成兩組，**左右各遞迴呼叫一次 `_grow`**，深度 + 1，結果掛到 `node.left`、`node.right`。

注意：只要有有效切點就切，就算 $\Delta I = 0$ 也照切（sklearn 也是：預設 `min_impurity_decrease=0`）。

**預測（`apply_one`）**：一筆樣本從根節點出發，在每個內部節點問一次「$x_j \le s$ 嗎？」，是就往左、否就往右，直到走到葉子。預測機率 = 葉子的類別比例 `value / n_samples`。

**特徵重要度**（impurity-based，sklearn 的 `feature_importances_`）：每個內部節點的「加權不純度下降」

$$n_t\, I(t) - n_{t_L} I(t_L) - n_{t_R} I(t_R)$$

加到它使用的特徵 $j$ 上，最後除以總和（所有特徵加起來 = 1）。

#### 符號表（3C）

| 符號 | 意義 | 程式裡 |
| --- | --- | --- |
| `Node` | 樹的節點：`n_samples`、`value`（各類筆數，$(K,)$）、`impurity`、`feature`、`threshold`、`left`、`right`、`depth` | `Node` |
| 葉子 | `left is None`（也就沒有 `right`）；`feature = -1` | `node.is_leaf` |
| idx | 落在這個節點的**訓練資料列號** | `idx`（$(n_t,)$） |
| `X_`、`y_idx_` | fit 時存下的整份訓練資料與類別編號 | `self.X_`、`self.y_idx_` |
| `root_` | 根節點；整棵樹透過 `left`／`right` 一層層掛在它下面 | `self.root_` |
| depth | 這個節點的深度（根 = 0） | `depth` |

#### 程式結構圖：誰身上有什麼

```
tree = DecisionTree(max_depth=2).fit(Xtr, ytr)
├── 超參數：criterion、max_depth、min_samples_split、min_samples_leaf
├── fit 存下的資料：X_ (N, d)、classes_ (K,)、y_idx_ (N,)、n_classes_、n_features_in_
└── root_ ─────────────► Node(n_samples=N, value=[…], feature=5, threshold=0.5, depth=0)
                          ├── left  ─► Node(…, feature=0, threshold=2.5, depth=1)
                          │            ├── left  ─► Node(葉子, depth=2)   left = right = None
                          │            └── right ─► Node(葉子, depth=2)
                          └── right ─► Node(…, depth=1)
                                       ├── left  ─► …
                                       └── right ─► …

_grow(idx, depth)：idx 是「這個節點的列號」，self.X_[idx]、self.y_idx_[idx] 才是這個節點的資料
apply_one(x)     ：node = self.root_，迴圈裡 node 一直被換成 node.left 或 node.right
```

`self` 永遠是那一棵樹（`DecisionTree`），不是節點；節點是另外的 `Node` 物件，用 `node.left` 這種屬性連起來。

#### 最小例子：TODO 8（遞迴長樹）

一維、6 筆：`x = [1, 2, 3, 4, 5, 6]`、`y = [0, 0, 1, 1, 1, 0]`、`max_depth = 2`。

1. **正確答案**：用手一層一層切（呼叫已經寫好的 `best_split`）：根節點切在哪、左右兩邊再各切在哪。
2. **積木**：一個節點上的終止條件、`best_split` 的回傳值、用遮罩把 `idx` 分成兩半 —— 子節點拿到的是「列號」。
3. **留給你**：`_grow` 裡左右兩次遞迴呼叫要傳什麼。

In [ ]:
# 最小例子：TODO 8（遞迴長樹）
X_mini = np.array([[1.0], [2.0], [3.0], [4.0], [5.0], [6.0]])
y_mini = np.array([0, 0, 1, 1, 1, 0])
dt_mini = DecisionTree(max_depth=2)
dt_mini.n_classes_ = 2

# ---- 1. 正確答案：一層一層手動切 ----
idx_root = np.arange(6)
g0, j0, s0 = dt_mini.best_split(X_mini[idx_root], y_mini[idx_root])
print(f"深度 0：idx={idx_root.tolist()}  y={y_mini[idx_root].tolist()}  → 切 x{j0} ≤ {s0}（ΔI={g0:.4f}）")
for side, mask in [("左", X_mini[idx_root, j0] <= s0), ("右", X_mini[idx_root, j0] > s0)]:
    idx_child = idx_root[mask]
    g1, j1, s1 = dt_mini.best_split(X_mini[idx_child], y_mini[idx_child])
    print(f"  深度 1（{side}）：idx={idx_child.tolist()}  y={y_mini[idx_child].tolist()}  Gini={float(gini(np.bincount(y_mini[idx_child], minlength=2))):.3f}", end="")
    print("  → 純的，當葉子" if gini(np.bincount(y_mini[idx_child], minlength=2)) == 0 else f"  → 切 x{j1} ≤ {s1}；深度 2 = max_depth，兩邊都是葉子")

# ---- 2. 積木 ----
print("\nbest_split 回傳 (ΔI, 特徵 j, 門檻 s)：", dt_mini.best_split(X_mini, y_mini))
go_left_mini = X_mini[idx_root, j0] <= s0
print("go_left = X_[idx, j] <= s  ", go_left_mini)
print("idx[go_left] =", idx_root[go_left_mini], " idx[~go_left] =", idx_root[~go_left_mini], "← 傳下去的是「列號」")
print("錯誤示範：X_mini[go_left] 是資料本身，不是列號；子節點要的是 idx，才能再用 self.X_[idx] 取資料")

# ---- 3. 留給你 ----
# 問題：_grow(idx, depth) 裡，node.left = self._grow(?, ?)、node.right = self._grow(?, ?)
# 寫完 TODO、執行過下兩格（第二格定義 print_tree）之後，取消下面的註解來對照（應該跟上面手動切的結果一樣）：
# print_tree(DecisionTree(max_depth=2).fit(X_mini, y_mini), ["x0"])

In [ ]:
from dataclasses import dataclass


@dataclass
class Node:
    """樹的一個節點。葉子：left = right = None、feature = −1。"""

    n_samples: int
    value: np.ndarray  # (K,) 這個節點裡各類的訓練樣本數
    impurity: float
    depth: int
    feature: int = -1
    threshold: float = np.nan
    left: "Node | None" = None
    right: "Node | None" = None

    @property
    def is_leaf(self) -> bool:
        return self.left is None


class DecisionTree(DecisionTree):  # 接續上一格的 class，補上長樹與預測
    def fit(self, X: ArrayLike, y: ArrayLike) -> Self:
        """X (N, d)、y (N,) → 長好整棵樹，存在 self.root_。"""
        self.X_ = np.asarray(X, dtype=float)
        self.classes_, self.y_idx_ = np.unique(np.asarray(y), return_inverse=True)
        self.n_classes_ = len(self.classes_)
        self.n_features_in_ = self.X_.shape[1]
        self.root_ = self._grow(np.arange(len(self.X_)), depth=0)
        return self

    def _grow(self, idx: np.ndarray, depth: int) -> Node:
        """用訓練資料的第 idx 列，長出一棵以這個節點為根的子樹，回傳這個節點。"""
        counts = np.bincount(self.y_idx_[idx], minlength=self.n_classes_).astype(float)
        node = Node(n_samples=len(idx), value=counts, impurity=float(self._impurity(counts)), depth=depth)
        # 終止條件：純的、樣本太少、深度到頂 → 葉子
        if (
            node.impurity <= 0
            or len(idx) < self.min_samples_split
            or (self.max_depth is not None and depth >= self.max_depth)
        ):
            return node
        gain, j, s = self.best_split(self.X_[idx], self.y_idx_[idx])
        if j < 0:  # 沒有任何有效切點（例如所有特徵都一樣、或 min_samples_leaf 擋住）
            return node
        node.feature, node.threshold = j, s
        go_left = self.X_[idx, j] <= s  # (n_t,) bool
        # TODO 8（程式）：遞迴長出左右子樹
        #   - 左邊拿 go_left 為 True 的列號、右邊拿其餘的；深度都是 depth + 1
        #   - 傳下去的是「列號」idx[...]，不是 self.X_[...] 的資料（看最小例子）
        raise NotImplementedError("TODO 8：遞迴")
        return node

In [ ]:
def print_tree(model: "DecisionTree", feature_names: list[str], max_depth: int | None = None) -> None:
    """把手寫樹印成類似 sklearn export_text 的樣子：每個葉子顯示 value（各類筆數）與預測類別。"""

    def walk(node: Node, prefix: str) -> None:
        if node.is_leaf or (max_depth is not None and node.depth >= max_depth):
            tag = "" if node.is_leaf else "（以下省略）"
            print(f"{prefix}value={node.value.astype(int).tolist()} → class {model.classes_[node.value.argmax()]}{tag}")
            return
        name = feature_names[node.feature]
        print(f"{prefix}{name} <= {node.threshold:.4g}")
        walk(node.left, prefix + "|   ")
        print(f"{prefix}{name} >  {node.threshold:.4g}")
        walk(node.right, prefix + "|   ")

    walk(model.root_, "")


def iter_nodes(model: "DecisionTree") -> list[Node]:
    """前序走訪（自己 → 左子樹 → 右子樹）回傳所有節點；順序跟 sklearn 的 tree_ 編號一致。"""
    out, stack = [], [model.root_]
    while stack:
        node = stack.pop()
        out.append(node)
        if not node.is_leaf:
            stack += [node.right, node.left]  # 後放的先拿：先處理左邊
    return out


tree_d2 = DecisionTree(max_depth=2).fit(Xtr, ytr)
print_tree(tree_d2, FEATURES)
print("節點數：", len(iter_nodes(tree_d2)), "（深度 2 的完整二元樹應 = 7）")

#### 最小例子：TODO 9（從根走到葉子）

用手建一棵 2 層的小樹（2 個內部節點：根、根的右子節點；3 片葉子），走一筆樣本 $x = (3, 0)$：

```
root: x0 <= 2.5 ?
├── 是 → 葉子 A  value=[4, 0]
└── 否 → x1 <= 0.5 ?
         ├── 是 → 葉子 B  value=[1, 3]
         └── 否 → 葉子 C  value=[2, 0]
```

1. **正確答案**：用 `if` 一層一層手寫：$x_0 = 3 > 2.5$ 往右、$x_1 = 0 \le 0.5$ 往左 → 葉子 B。
2. **積木**：`node.is_leaf`、`x[node.feature] <= node.threshold`、`node = node.left` 是「換一個節點來看」。
3. **留給你**：把「一直往下走直到葉子」寫成 `while` 迴圈。

In [ ]:
# 最小例子：TODO 9（從根走到葉子）
leaf_A = Node(4, np.array([4.0, 0.0]), 0.0, depth=1)
leaf_B = Node(4, np.array([1.0, 3.0]), 0.375, depth=2)
leaf_C = Node(2, np.array([2.0, 0.0]), 0.0, depth=2)
right_mini = Node(6, np.array([3.0, 3.0]), 0.5, depth=1, feature=1, threshold=0.5, left=leaf_B, right=leaf_C)
root_mini = Node(10, np.array([7.0, 3.0]), 0.42, depth=0, feature=0, threshold=2.5, left=leaf_A, right=right_mini)
x_q_mini = np.array([3.0, 0.0])

# ---- 1. 正確答案：手寫兩層 if ----
if x_q_mini[0] <= 2.5:
    ans_mini = leaf_A
elif x_q_mini[1] <= 0.5:
    ans_mini = leaf_B
else:
    ans_mini = leaf_C
print("正確答案：走到 value =", ans_mini.value, " 機率 =", ans_mini.value / ans_mini.n_samples)

# ---- 2. 積木 ----
node_mini = root_mini
print(f"\n現在在根：is_leaf={node_mini.is_leaf}  問 x[{node_mini.feature}] = {x_q_mini[node_mini.feature]} <= {node_mini.threshold}？", x_q_mini[node_mini.feature] <= node_mini.threshold)
node_mini = node_mini.right
print(f"node = node.right 之後：is_leaf={node_mini.is_leaf}  問 x[{node_mini.feature}] = {x_q_mini[node_mini.feature]} <= {node_mini.threshold}？", x_q_mini[node_mini.feature] <= node_mini.threshold)
print("root_mini 有沒有被改到？", root_mini.feature, root_mini.threshold, "← node = node.right 只是讓變數 node 指向另一個物件，樹本身不變")

# ---- 3. 留給你 ----
# 問題：while 迴圈的條件是什麼？迴圈裡 node 要換成 node.left 還是 node.right，由哪個比較決定？
# 寫完 TODO、執行過下一格之後，取消下面的註解來對照：
# t_mini = DecisionTree(); t_mini.root_ = root_mini; print(t_mini.apply_one(x_q_mini).value)

In [ ]:
class DecisionTree(DecisionTree):  # 接續：預測
    def apply_one(self, x: np.ndarray) -> Node:
        """一筆樣本 x (d,) → 它落到的葉子 Node。"""
        node = self.root_
        # TODO 9（程式）：從根一路走到葉子
        #   - 還不是葉子（not node.is_leaf）就繼續走
        #   - x[node.feature] <= node.threshold 往左（node.left），否則往右（node.right）
        raise NotImplementedError("TODO 9：走到葉子")
        return node

    def predict_proba(self, X: ArrayLike) -> np.ndarray:
        """X (n, d) → (n, K)：落到的葉子的類別比例（欄的順序 = classes_）。"""
        X = np.asarray(X, dtype=float)
        leaves = [self.apply_one(x) for x in X]
        return np.array([leaf.value / leaf.n_samples for leaf in leaves])

    def predict(self, X: ArrayLike) -> np.ndarray:
        """X (n, d) → (n,) 的類別標籤。argmax 平手時取第一個（編號小的類別），sklearn 也是。"""
        return self.classes_[self.predict_proba(X).argmax(axis=1)]

    def get_depth(self) -> int:
        return max(node.depth for node in iter_nodes(self))

    def get_n_leaves(self) -> int:
        return sum(node.is_leaf for node in iter_nodes(self))

#### 最小例子：TODO 10（特徵重要度）

用上一個最小例子的手建小樹（根用特徵 0、右邊的內部節點用特徵 1），Gini 已經存在每個節點的 `impurity`。

1. **正確答案**：兩個內部節點各算一次 $n_t I(t) - n_L I(t_L) - n_R I(t_R)$，加到各自的特徵上，再正規化。目標 `[0.45, 0.55]`（四捨五入）。
2. **積木**：`iter_nodes` 走過所有節點、跳過葉子；`np.zeros(d)` 當累加器，`imp[node.feature] += ...`。
3. **留給你**：寫出那一行累加。

In [ ]:
# 最小例子：TODO 10（特徵重要度）
# 先把手建小樹的不純度改成真正的 Gini（上面為了好讀寫的是近似值）
for nd in [leaf_A, leaf_B, leaf_C, right_mini, root_mini]:
    nd.impurity = float(gini(nd.value))

# ---- 1. 正確答案：逐個內部節點手算 ----
dec_root = 10 * root_mini.impurity - 4 * leaf_A.impurity - 6 * right_mini.impurity
dec_right = 6 * right_mini.impurity - 4 * leaf_B.impurity - 2 * leaf_C.impurity
imp_loop = np.array([dec_root, dec_right])
print(f"根（特徵 0）：10·{root_mini.impurity:.3f} − 4·{leaf_A.impurity:.3f} − 6·{right_mini.impurity:.3f} = {dec_root:.3f}")
print(f"右（特徵 1）： 6·{right_mini.impurity:.3f} − 4·{leaf_B.impurity:.3f} − 2·{leaf_C.impurity:.3f} = {dec_right:.3f}")
print("正確答案（正規化後）：", (imp_loop / imp_loop.sum()).round(4))

# ---- 2. 積木 ----
t_mini = DecisionTree()
t_mini.root_ = root_mini
t_mini.n_features_in_ = 2
for nd in iter_nodes(t_mini):
    print(f"  depth={nd.depth} is_leaf={nd.is_leaf!s:<5} feature={nd.feature:>2} n={nd.n_samples:>2} impurity={nd.impurity:.3f}")

# ---- 3. 留給你 ----
# 問題：for node in iter_nodes(self): 跳過葉子之後，imp[node.feature] += ?（用 node、node.left、node.right 的 n_samples 與 impurity）
# 寫完 TODO、執行過下一格之後，取消下面的註解來對照：
# t2_mini = DecisionTree(); t2_mini.root_ = root_mini; t2_mini.n_features_in_ = 2; print(t2_mini.feature_importances_.round(4))

In [ ]:
class DecisionTree(DecisionTree):  # 接續：特徵重要度
    @property
    def feature_importances_(self) -> np.ndarray:
        """(d,)：每個特徵的加權不純度下降總和，正規化成總和 1。"""
        imp = np.zeros(self.n_features_in_)
        for node in iter_nodes(self):
            if node.is_leaf:
                continue
            # TODO 10（程式）：把這個內部節點的「加權不純度下降」加到它用的特徵上
            #   - n_t·I(t) − n_L·I(t_L) − n_R·I(t_R)，n 用 n_samples、I 用 impurity
            raise NotImplementedError("TODO 10：特徵重要度")
        total = imp.sum()
        return imp / total if total > 0 else imp

In [ ]:
from sklearn.tree import DecisionTreeClassifier


def same_structure(mine: DecisionTree, sk: DecisionTreeClassifier) -> bool:
    """兩棵樹的每個節點（前序順序）是否用同一個特徵、同一個門檻、同樣的類別筆數。"""
    nodes = iter_nodes(mine)
    t = sk.tree_
    if len(nodes) != t.node_count:
        return False
    for i, nd in enumerate(nodes):
        sk_leaf = t.children_left[i] == -1
        if nd.is_leaf != sk_leaf:
            return False
        if not sk_leaf and (nd.feature != t.feature[i] or not np.isclose(nd.threshold, t.threshold[i])):
            return False
        if not np.allclose(nd.value / nd.n_samples, t.value[i].ravel() / t.value[i].sum()):
            return False
    return True


for crit in ["gini", "entropy"]:
    for depth in [1, 2, 3, 4]:
        mine = DecisionTree(criterion=crit, max_depth=depth).fit(Xtr, ytr)
        sk = DecisionTreeClassifier(criterion=crit, max_depth=depth, random_state=0).fit(Xtr, ytr)
        print(
            f"[{crit:<7} depth={depth}] ✅ 結構相同：{same_structure(mine, sk)!s:<5}",
            f"predict_proba 最大差異：{np.abs(mine.predict_proba(Xte) - sk.predict_proba(Xte)).max():.1e}",
            f"importance 最大差異：{np.abs(mine.feature_importances_ - sk.feature_importances_).max():.1e}",
            f"test acc {np.mean(mine.predict(Xte) == yte):.4f}",
        )
print("（深度 1–4 應該結構完全相同、差異 < 1e-12）")

In [ ]:
# 長深一點：手寫 vs sklearn，以及「sklearn 換一個 random_state」
def same_sk(a: DecisionTreeClassifier, b: DecisionTreeClassifier) -> bool:
    """兩棵 sklearn 樹的結構（節點數、每個節點的特徵與門檻）是否完全相同。"""
    return a.tree_.node_count == b.tree_.node_count and np.array_equal(a.tree_.feature, b.tree_.feature) and np.allclose(a.tree_.threshold, b.tree_.threshold)


rows = []
for depth in [5, 6, 8, None]:
    mine = DecisionTree(max_depth=depth).fit(Xtr, ytr)
    sks = [DecisionTreeClassifier(max_depth=depth, random_state=s).fit(Xtr, ytr) for s in range(10)]
    rows.append(
        {
            "max_depth": str(depth),
            "mine: leaves": mine.get_n_leaves(),
            "sklearn: leaves (10 seeds)": sorted({m.get_n_leaves() for m in sks}),
            "mine == sklearn(seed 0)": same_structure(mine, sks[0]),
            "sklearn 10 個 seed 的結構都相同": all(same_sk(sks[0], m) for m in sks[1:]),
            "mine train acc": np.mean(mine.predict(Xtr) == ytr),
            "sklearn train acc": sks[0].score(Xtr, ytr),
            "mine test acc": np.mean(mine.predict(Xte) == yte),
            "sklearn test acc (10 seeds)": sorted({round(m.score(Xte, yte), 4) for m in sks}),
        }
    )
display(pd.DataFrame(rows).set_index("max_depth"))


def count_tied_nodes(model: DecisionTree) -> tuple[int, int]:
    """(最好的切法不唯一的內部節點數, 內部節點總數)。「不唯一」= 有兩個以上特徵的最佳 ΔI 一樣大。"""
    tied = total = 0
    stack = [(model.root_, np.arange(len(model.X_)))]
    while stack:
        node, idx = stack.pop()
        if node.is_leaf:
            continue
        total += 1
        gains = np.array(
            [model.best_split_on_feature(model.X_[idx, j], model.y_idx_[idx], model.n_classes_)[0] for j in range(model.n_features_in_)]
        )
        tied += int((np.abs(gains - gains.max()) < 1e-12).sum() > 1)
        go = model.X_[idx, node.feature] <= node.threshold
        stack += [(node.left, idx[go]), (node.right, idx[~go])]
    return tied, total


for depth in [4, 5, 8]:
    t, n = count_tied_nodes(DecisionTree(max_depth=depth).fit(Xtr, ytr))
    print(f"max_depth={depth}：{n} 個內部節點中，{t} 個的最佳切法有平手（不只一個特徵）")

In [ ]:
# TODO 11（問答）手寫 vs sklearn
#
# (1) 深度 1–4 完全一致，更深之後葉子數、測試 accuracy 開始和 sklearn 不同，而且 sklearn 換 random_state 也會變。
#     (a) 讀上一格：深度 4 有幾個平手的節點？深度 5、8 呢？
#     (b) sklearn 每個節點會把 d 個特徵「隨機打亂順序」再逐一檢查，遇到 ΔI 一樣大時保留先看到的那個。
#         手寫版則固定從特徵 0 看到特徵 8。為什麼這會讓兩者在平手時選到不同的特徵？
#         選到不同的特徵之後，為什麼影響會一路傳到下面所有的子樹？
#     (c) 哪些特徵容易平手？（提示：特徵都是整數或 0／1，節點很小時，兩種切法可能把樣本分成「完全一樣的兩群」）
#     (d) 對照第四章 TODO 8 的距離平手：那裡為什麼「預測」一致、這裡為什麼連預測都不一樣？
# 答：
#
# (2) 完全長滿（max_depth=None）的樹，train accuracy 為什麼不是 1.0？
#     （提示：如果兩個乘客 9 個特徵完全一樣、一個生還一個罹難，樹能不能把他們分開？用下面這段程式驗證）
#       dup = Z_train.assign(y=ytr).groupby(FEATURES).y.agg(["nunique", "size"])
#       print("特徵完全相同、標籤卻不同的組數：", (dup["nunique"] > 1).sum())
# 答：
#
# (3) predict_proba 一次只走一筆（Python 迴圈），sklearn 快很多。
#     預測一筆的成本是 O(?)（用樹的深度表示）？跟第四章 KNN 預測一筆的 O(n_train · d) 比呢？
# 答：

### 3D. 實驗：不純度的種類、深度、單調轉換、不穩定性

#### 本節名詞

| 名詞 | 定義 |
| --- | --- |
| 誤分類率 | $E = 1 - \max_k p_k$：節點裡「猜多數」會猜錯的比例 |
| 嚴格凹 | 圖形往上拱、而且沒有直線段；Gini 和 entropy 是，誤分類率在 $p < 0.5$、$p > 0.5$ 兩段各是直線 |
| resubstitution accuracy | 在訓練集上評估（第四章 3C） |
| 單調轉換 | 不改變排序的轉換（前置複習 6），例如 $\log(1 + x)$、標準化 |
| bootstrap 樹 | 在 bootstrap 樣本（抽後放回 $N$ 次，前置複習 7）上長的樹 |
| 不一致率（disagreement） | 兩棵樹在測試集上預測不同的比例 |

In [ ]:
# 實驗 1：為什麼不用誤分類率來挑切點？
# 一個 800 筆、各 400 的節點，兩種切法：
#   切法 A：(300, 100) | (100, 300)
#   切法 B：(200, 400) | (200, 0)      ← 右邊是純的
parent = np.array([400, 400])
splits = {"A": (np.array([300, 100]), np.array([100, 300])), "B": (np.array([200, 400]), np.array([200, 0]))}
rows = []
for name, (cl, cr) in splits.items():
    row = {"split": name, "left": cl.tolist(), "right": cr.tolist()}
    for crit_name, f in [("misclassification", misclassification), ("gini", gini), ("entropy", entropy)]:
        nl, nr = cl.sum(), cr.sum()
        row[f"Δ {crit_name}"] = float(f(parent) - (nl * f(cl) + nr * f(cr)) / (nl + nr))
    rows.append(row)
display(pd.DataFrame(rows).set_index("split").round(4))

# 在 Titanic 上：三種 criterion 的 CV accuracy（sklearn 沒有 misclassification，只比 gini 與 entropy）
crit_tbl = pd.DataFrame(
    {
        crit: [cross_val_score(DecisionTreeClassifier(criterion=crit, max_depth=d, random_state=0), Xtr, ytr, cv=cv).mean() for d in [2, 3, 4, 6, 8, None]]
        for crit in ["gini", "entropy"]
    },
    index=["2", "3", "4", "6", "8", "None"],
).rename_axis("max_depth")
display(crit_tbl.round(4))

# 兩種 criterion 長出來的樹，前幾層一樣嗎？
for d in [2, 3, 4]:
    tg = DecisionTreeClassifier(criterion="gini", max_depth=d, random_state=0).fit(Xtr, ytr)
    te = DecisionTreeClassifier(criterion="entropy", max_depth=d, random_state=0).fit(Xtr, ytr)
    print(f"max_depth={d}：gini 與 entropy 的測試集預測一致比例 = {(tg.predict(Xte) == te.predict(Xte)).mean():.4f}")

In [ ]:
# TODO 12（問答）不純度的種類
#
# (1) 讀實驗 1 的表：
#     (a) 用誤分類率，切法 A、B 的 ΔI 各是多少？它能分辨兩者嗎？
#     (b) 用 Gini、entropy，哪一個切法比較好？直覺上你會選哪一個？（提示：B 的右邊已經是純的，之後不用再切）
#     (c) 誤分類率的問題：它只看「多數類」，在 p < 0.5 和 p > 0.5 各是一段直線。
#         切法 A、B 的兩個子節點都落在同一段直線上（都還是「左邊多數 = 0」？還是不一定？自己檢查）時，
#         子節點的加權平均剛好落在那條直線上 → ΔI 會是多少？
#     (d) 用前置複習 5（凹函數）說明：為什麼 Gini、entropy 的 ΔI 永遠 ≥ 0，而且只要左右兩邊的比例不一樣就 > 0？
#         （提示：父節點的比例 = 左右比例的加權平均；函數在「平均」的值 ≥ 函數值的平均）
# 答：
#
# (2) 誤分類率的 ΔI 常常是 0（看不出進步），對「貪婪地一層一層長」的樹有什麼後果？
#     （提示：如果第一刀用誤分類率看起來沒有進步，但它是讓第二刀變得很好的那一刀呢？）
#     那為什麼剪枝（第 5 節）和最後評估時，又可以用 accuracy（= 1 − 誤分類率）？
# 答：
#
# (3) 看三條曲線的圖與 Titanic 的表：
#     (a) entropy / 2 的曲線和 Gini 長得多像？
#     (b) gini 與 entropy 的 CV accuracy 差多少？測試集預測一致的比例有多高？
#     (c) 結論：criterion 值得花時間調嗎？
# 答：

In [ ]:
# 實驗 2：深度。train（resubstitution）vs 5-fold CV accuracy，以及葉子數
depths = list(range(1, 16)) + [None]
rows = []
for d in depths:
    m = DecisionTreeClassifier(max_depth=d, random_state=0)
    s = cross_val_score(m, Xtr, ytr, cv=cv)
    m.fit(Xtr, ytr)
    rows.append({"max_depth": d if d is not None else 99, "train": m.score(Xtr, ytr), "cv": s.mean(), "cv_std": s.std(), "leaves": m.get_n_leaves(), "actual depth": m.get_depth()})
depth_tbl = pd.DataFrame(rows).set_index("max_depth")

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
pos = np.arange(len(depth_tbl))  # x 軸用位置，標籤另外寫（None 放最右邊）
x_lab = [str(d) if d != 99 else "None" for d in depth_tbl.index]
sex_rule_cv = float(baseline.loc["Sex rule", "accuracy"].split(" ")[0])
axes[0].plot(pos, depth_tbl["train"], "o-", label="train (resubstitution)")
axes[0].plot(pos, depth_tbl["cv"], "o-", label="5-fold CV")
axes[0].fill_between(pos, depth_tbl["cv"] - depth_tbl["cv_std"], depth_tbl["cv"] + depth_tbl["cv_std"], alpha=0.2, color="C1")
axes[0].axhline(sex_rule_cv, color="gray", ls=":", label="sex rule (CV)")
axes[0].set_xlabel("max_depth (larger = more complex)")
axes[0].set_ylabel("accuracy")
axes[0].legend()
axes[0].set_title("Effect of max_depth")
axes[1].plot(pos, depth_tbl["leaves"], "o-")
axes[1].set_yscale("log")
axes[1].set_xlabel("max_depth")
axes[1].set_ylabel("number of leaves (log)")
axes[1].set_title("Tree size")
for ax in axes:
    ax.set_xticks(pos, x_lab)
fig.tight_layout()
plt.show()
display(depth_tbl.round(4).T)

leaf_sizes = np.array([nd.n_samples for nd in iter_nodes(DecisionTree().fit(Xtr, ytr)) if nd.is_leaf])
print(f"完全長滿的手寫樹：{len(leaf_sizes)} 個葉子，其中 {np.mean(leaf_sizes == 1):.0%} 只有 1 筆樣本、{np.mean(leaf_sizes <= 3):.0%} ≤ 3 筆")

In [ ]:
# TODO 13（問答）深度
#
# (1) (a) 讀表：max_depth = 1、3、None 時 train 與 cv 各是多少？差距怎麼變？
#     (b) 哪一段是欠擬合、哪一段是過擬合？過擬合在圖的左邊還是右邊？
#         跟第三章 C（右邊過擬合）、第四章 k（左邊過擬合）比，決策樹的「複雜」方向和哪一個一樣？
#     (c) max_depth = 1 的 CV 和性別規則幾乎一樣，為什麼？（TODO 7）
# 答：
#
# (2) max_depth 很大時，有好多 max_depth 的「實際深度」和葉子數都一樣（看 actual depth、leaves）。為什麼？
#     （提示：樹自己停下來的條件是什麼？）
# 答：
#
# (3) 完全長滿的樹，很多葉子只有 1 筆樣本。
#     (a) 這種葉子的 predict_proba 會是多少？用它來預測一位新乘客，可信嗎？
#     (b) 用「深度 3 的樹 vs 長滿的樹」各說一句：誰的偏差高、誰的變異高？
#     (c) 除了 max_depth，還有什麼方法能防止「1 筆一個葉子」？（看 DecisionTreeClassifier 的參數；第 5 節會調）
# 答：

In [ ]:
# 實驗 3：單調轉換。標準化、fare 取 log、所有特徵換成「排名」——手寫樹的預測會變嗎？
from sklearn.preprocessing import StandardScaler
from scipy.stats import rankdata

j_fare = FEATURES.index("fare")
transforms = {}
sc = StandardScaler().fit(Xtr)
transforms["standardize"] = (sc.transform(Xtr), sc.transform(Xte))
L_tr, L_te = Xtr.copy(), Xte.copy()
L_tr[:, j_fare], L_te[:, j_fare] = np.log1p(Xtr[:, j_fare]), np.log1p(Xte[:, j_fare])
transforms["log1p(fare)"] = (L_tr, L_te)
R_tr = np.column_stack([rankdata(Xtr[:, j], method="dense") for j in range(Xtr.shape[1])]).astype(float)
# 測試集：用「訓練集裡 ≤ 它的不同值有幾個」當排名（只用訓練集的資訊）
R_te = np.column_stack([np.searchsorted(np.unique(Xtr[:, j]), Xte[:, j], side="right") for j in range(Xtr.shape[1])]).astype(float)
transforms["rank (all features)"] = (R_tr, R_te)

base = DecisionTree(max_depth=None).fit(Xtr, ytr)
rows = []
for name, (A_tr, A_te) in transforms.items():
    m = DecisionTree(max_depth=None).fit(A_tr, ytr)
    rows.append(
        {
            "transform": name,
            "訓練集 predict_proba 完全相同": bool(np.array_equal(base.predict_proba(Xtr), m.predict_proba(A_tr))),
            "每個節點用的特徵都相同": [n.feature for n in iter_nodes(base)] == [n.feature for n in iter_nodes(m)],
            "測試集預測一致比例": np.mean(base.predict(Xte) == m.predict(A_te)),
        }
    )
display(pd.DataFrame(rows).set_index("transform"))

# 對照：同樣的轉換對 KNN（第四章）
from sklearn.neighbors import KNeighborsClassifier

knn_rows = {}
for name, (A_tr, _) in {"raw": (Xtr, None), **transforms}.items():
    knn_rows[name] = cross_val_score(KNeighborsClassifier(n_neighbors=15), A_tr, ytr, cv=cv).mean()
print("KNN（k=15）在各種轉換下的 CV accuracy：", {k: round(v, 4) for k, v in knn_rows.items()})

In [ ]:
# TODO 14（問答）單調轉換
#
# (1) (a) 讀表：三種轉換之後，訓練集的預測、每個節點選的特徵有沒有改變？
#     (b) 決策樹每一刀只問「x_j ≤ s 嗎？」。用前置複習 6 解釋：單調遞增轉換之後，
#         同一個切點左右兩邊的「人」會不會變？ΔI 會不會變？所以選到的切法會不會變？
#     (c) 門檻 s 本身會不會變？（例如 fare 取 log 之後，門檻 50 變成多少？）
# 答：
#
# (2) 訓練集保證完全一致，測試集卻不保證：rank 那一列不是 1.0。
#     (a) 門檻取兩個訓練值的「中點」。fare 的兩個相鄰訓練值 10 和 20，原始中點是 15；
#         取 log1p 之後的中點，換回原本的尺度是多少？一位票價 14.5 的新乘客在兩棵樹裡會不會走到不同的邊？
#     (b) rank 轉換把測試值換成「訓練集裡 ≤ 它的不同值有幾個」：一個落在兩個訓練值「之間」的測試值，
#         會被當成和哪一個訓練值一樣？這等於把門檻從中點移到哪裡？
#     (c) log1p(fare) 那一列剛好是 1.0，是理論保證還是這份資料剛好如此？
# 答：
#
# (3) 對照第四章 TODO 11：KNN 的「縮放 = 改變距離的權重」，標準化會改變預測；決策樹呢？
#     由此回答 TODO 1 (4)：決策樹需要標準化或取 log 嗎？
#     那「特徵縮放」對決策樹完全沒有影響嗎？（提示：如果是兩個特徵相加 x_1 + x_2 的新特徵呢？那不是單調轉換「一個」特徵）
# 答：

In [ ]:
# 實驗 4：不穩定性。在 20 個 bootstrap 樣本上各長一棵完全長滿的樹
rng = np.random.default_rng(0)
B = 20
boot_proba, first_splits = [], []
for b in range(B):
    i_boot = rng.integers(0, len(Xtr), len(Xtr))  # 抽後放回 N 次
    t = DecisionTreeClassifier(random_state=0).fit(Xtr[i_boot], ytr[i_boot])
    boot_proba.append(t.predict_proba(Xte)[:, 1])
    tr_ = t.tree_
    l, r = tr_.children_left[0], tr_.children_right[0]
    first_splits.append(
        f"root: {FEATURES[tr_.feature[0]]}"
        f" | L: {FEATURES[tr_.feature[l]]} <= {tr_.threshold[l]:.4g}"
        f" | R: {FEATURES[tr_.feature[r]]} <= {tr_.threshold[r]:.4g}"
    )
boot_proba = np.array(boot_proba)  # (B, n_test)
boot_pred = (boot_proba > 0.5).astype(int)

print("前兩層的切法（出現次數）：")
display(pd.Series(first_splits).value_counts().to_frame("count"))

pair_dis = [np.mean(boot_pred[a] != boot_pred[b]) for a in range(B) for b in range(a + 1, B)]
single_acc = (boot_pred == yte).mean(axis=1)
bag_acc = np.mean((boot_proba.mean(axis=0) > 0.5) == yte)
print(f"兩兩之間的測試集預測不一致率：平均 {np.mean(pair_dis):.3f}（最小 {np.min(pair_dis):.3f}、最大 {np.max(pair_dis):.3f}）")
print(f"單棵樹的 test accuracy：平均 {single_acc.mean():.4f} ± {single_acc.std():.4f}")
print(f"20 棵樹的機率平均起來再預測（bagging）：test accuracy {bag_acc:.4f}")

fig, ax = plt.subplots(figsize=(7, 3.5))
ax.hist(boot_pred.mean(axis=0), bins=np.linspace(0, 1, 21))
ax.set_xlabel("fraction of the 20 trees predicting 'survived'")
ax.set_ylabel("test passengers")
ax.set_title("How much do bootstrap trees agree?")
plt.show()

In [ ]:
# TODO 15（問答）不穩定性
#
# (1) (a) 讀輸出：20 棵樹的根節點都一樣嗎？第二層呢？
#     (b) 為什麼「根節點」很穩定，越往下越不穩定？（提示：TODO 7 (2) 根節點第一名與第二名的 ΔI 差距；
#         越往下，每個節點的樣本越少 —— 少幾個人對「哪個門檻最好」的影響是大還是小？）
#     (c) 上一層選了不同的切法，下面整棵子樹會怎樣？（TODO 11 (1)(b) 的同一個機制）
# 答：
#
# (2) 兩棵樹的測試集預測平均有多少比例不同？這是偏差—變異裡的哪一個？
#     跟 KNN（k 很小時）比：KNN 換一批訓練資料，預測會變很多嗎？哪一個比較「不穩定」？
# 答：
#
# (3) 20 棵樹的機率平均起來（bagging）之後，test accuracy 比單棵樹的平均高還是低？
#     (a) 「平均」降低的是偏差還是變異？（提示：n 個獨立、變異都是 σ² 的數，平均的變異是多少？）
#     (b) 這 20 棵樹不是獨立的（都用同一份資料抽樣、根節點都一樣）。如果想讓它們「更不像」，可以怎麼做？
#         （這就是第 10 章 Random Forest 的「隨機特徵」：每個節點只從隨機挑的幾個特徵裡選）
# 答：

---
## 4. 套件實作：sklearn `DecisionTreeClassifier`

前處理和樹包進同一個 `Pipeline`：CV 的每個 fold 都會重新 fit 補值與編碼（只看那個 fold 的訓練部分）。決策樹不需要標準化（實驗 3），所以沒有 scaler。

#### 本節名詞

| 名詞 | 意思 |
| --- | --- |
| `tree_` | sklearn 把樹存成好幾個平行的陣列：`feature[i]`、`threshold[i]`、`children_left[i]`、`children_right[i]`、`value[i]`、`n_node_samples[i]`、`impurity[i]`，第 $i$ 個節點的資訊在各陣列的第 $i$ 格；葉子的 `children_left = -1` |
| `export_text` | 把樹印成文字規則（跟 `print_tree` 一樣） |
| `plot_tree` | 把樹畫出來；顏色越深代表越純 |
| `apply(X)` | 每一筆落在哪個葉子（`tree_` 的節點編號） |
| impurity-based importance | `feature_importances_`：TODO 10 的算法，**用訓練資料**算 |
| permutation importance | 把驗證／測試資料的某一欄**隨機打亂**，看分數掉多少；掉越多越重要（第 7 節） |

#### 程式結構圖

```
tree_pipe = Pipeline([("prep", make_prep()), ("tree", DecisionTreeClassifier(...))])
├── tree_pipe["prep"]：ColumnTransformer，原始 7 欄 → 9 個數值欄（get_feature_names_out() 拿欄名）
└── tree_pipe["tree"]：DecisionTreeClassifier；fit 後有 tree_、feature_importances_、get_depth()、get_n_leaves()
參數名：tree__max_depth、tree__min_samples_leaf、tree__ccp_alpha、tree__criterion
```

In [ ]:
from sklearn.tree import export_text, plot_tree

tree_pipe = Pipeline([("prep", make_prep()), ("tree", DecisionTreeClassifier(max_depth=3, random_state=0))])
tree_pipe.fit(X_train, y_train)

scratch_d3 = DecisionTree(max_depth=3).fit(Xtr, ytr)
print("✅ 手寫 vs sklearn Pipeline：predict 一致比例", np.mean(scratch_d3.predict(Xte) == tree_pipe.predict(X_test)), "（應 = 1.0）")
print("✅ predict_proba 最大差異：", np.abs(scratch_d3.predict_proba(Xte) - tree_pipe.predict_proba(X_test)).max(), "（應 < 1e-12）")
print("✅ 結構相同：", same_structure(scratch_d3, tree_pipe["tree"]))
print()
print(export_text(tree_pipe["tree"], feature_names=FEATURES, show_weights=True))

fig, ax = plt.subplots(figsize=(18, 7))
plot_tree(tree_pipe["tree"], feature_names=FEATURES, class_names=["died", "survived"], filled=True, rounded=True, fontsize=8, ax=ax)
plt.show()

In [ ]:
# 每個葉子是一條規則：條件、人數、生還率
t = tree_pipe["tree"].tree_


def leaf_rules(t, feature_names: list[str]) -> pd.DataFrame:
    """sklearn 的 tree_ → 每個葉子一列：規則字串、訓練樣本數、生還率。"""
    rows = []

    def walk(i: int, conds: list[str]) -> None:
        if t.children_left[i] == -1:
            v = t.value[i].ravel()
            rows.append({"leaf": i, "rule": " & ".join(conds) or "(root)", "n": int(t.n_node_samples[i]), "survival rate": v[1] / v.sum()})
            return
        f, s = feature_names[t.feature[i]], t.threshold[i]
        walk(t.children_left[i], conds + [f"{f} <= {s:.4g}"])
        walk(t.children_right[i], conds + [f"{f} > {s:.4g}"])

    walk(0, [])
    return pd.DataFrame(rows).set_index("leaf")


rules = leaf_rules(t, FEATURES)
display(rules.round(3))
print("predict_proba 在測試集上出現的不同值：", np.unique(tree_pipe.predict_proba(X_test)[:, 1].round(4)))

imp_tbl = pd.DataFrame(
    {"sklearn (depth 3)": tree_pipe["tree"].feature_importances_, "scratch (depth 3)": scratch_d3.feature_importances_},
    index=FEATURES,
).sort_values("sklearn (depth 3)", ascending=False)
display(imp_tbl.round(4))

In [ ]:
# TODO 16（問答）讀一棵樹
#
# (1) 把深度 3 的樹翻成人話：
#     (a) 生還率最高、最低的葉子各是哪一群人？各有幾個人？
#         有兩片葉子的生還率很低，條件裡都有「sibsp 大」或「fare 高、但是三等艙」—— 這可能是什麼樣的乘客？
#     (b) 女性那一邊，第二刀切的是什麼？男性那一邊呢？對照 EDA 的交叉表與年齡分組，這兩刀合理嗎？
#     (c) 男性那一邊的第二刀切在 age（template 測試時是 age <= 12.5，以你的輸出為準）。用 EDA（以及 name 裡的稱謂 Master）解釋這群人是誰；
#         為什麼這一刀只出現在男性那邊、不在女性那邊？（提示：決策樹自動處理了「交互作用」——
#         某個特徵的效果在不同組別裡不一樣；第三章的邏輯回歸要怎麼做才能表達同樣的事？）
# 答：
#
# (2) predict_proba 在測試集上只有幾種不同的值？為什麼？
#     這對 ROC 曲線、或「用 0.5 以外的門檻」做決策有什麼影響？跟第四章 KNN 的機率（只能是 1/k 的倍數）比較。
# 答：
#
# (3) 特徵重要度：
#     (a) 最重要的特徵是哪個？占多少？age、fare 這兩個連續特徵的重要度高嗎？
#     (b) TODO 10 的算法只看「訓練資料上不純度降了多少」。如果一個特徵有很多不同的值（例如 fare、age），
#         它有很多候選門檻可以挑 —— 就算它跟生還完全無關，也比較容易「碰巧」找到一刀讓不純度下降一點。
#         這會讓重要度偏向哪一種特徵？（第 7 節會加一個純雜訊的特徵來驗證）
# 答：

---
## 5. 調參

全程只用 `X_train`（測試集鎖住），依 README「調參流程」：

| 類別 | 超參數 | 說明 |
| --- | --- | --- |
| 容量 | `max_depth` | 深度上限；越大越複雜 |
| 容量 | `min_samples_leaf` | 每個葉子至少幾筆；越大越簡單（防止「1 筆一個葉子」） |
| 容量 | `min_samples_split` | 至少幾筆才考慮切；跟 `min_samples_leaf` 作用相似 |
| 容量 | `max_leaf_nodes` | 葉子數上限（改成「最好的先長」的長法） |
| 正則化 | `ccp_alpha` | 成本複雜度剪枝：先長滿、再剪掉「不划算」的子樹；越大剪越多 |
| 不純度 | `criterion` | gini／entropy；實驗 1 看過幾乎沒差，不調 |

#### 成本複雜度剪枝（cost-complexity pruning，CART 的剪枝法）

對一棵樹 $T$ 定義

$$R_\alpha(T) = R(T) + \alpha\,|T|$$

- $R(T)$：所有葉子的「加權不純度」總和 $\sum_{\text{葉子 } \ell} \frac{n_\ell}{N} I(\ell)$（越小越純）；
- $|T|$：葉子數（越多越複雜）；
- $\alpha \ge 0$：每多一片葉子要付的代價。

$\alpha = 0$ 時完全長滿的樹最好；$\alpha$ 變大時，把某個子樹整個剪成一片葉子會讓 $R$ 上升一點、但 $|T|$ 下降很多，就變得划算。
`cost_complexity_pruning_path` 會算出一串「剛好讓某個子樹被剪掉」的 $\alpha$ 值（從小到大），每個 $\alpha$ 對應一棵越來越小的樹，最後一個 $\alpha$ 剪到只剩根節點。

#### 本節名詞

- **validation curve**（`validation_curve`）：只改一個超參數，每個值做一次 CV，畫 train 與 validation 兩條曲線。
- **grid search**（`GridSearchCV`）：試完 `param_grid` 所有組合；結果在 `cv_results_`、`best_params_`、`best_estimator_`。
- **1-SE rule**：門檻 = 最佳那組的平均 − 最佳那組的標準差；平均 $\ge$ 門檻的都算「跟最佳分不出來」，從中挑最簡單的。決策樹的「簡單」= 葉子少 = $\alpha$ 大。
- **learning curve**（`learning_curve`）：訓練資料由少到多，看多給資料有沒有幫助。

In [ ]:
from sklearn.model_selection import GridSearchCV, learning_curve, validation_curve

default_pipe = Pipeline([("prep", make_prep()), ("tree", DecisionTreeClassifier(random_state=0))])
baseline.loc["Tree default (fully grown)"] = cv_summary(default_pipe, X_train, y_train)
baseline.loc["Tree depth 3"] = cv_summary(tree_pipe, X_train, y_train)
display(baseline)

In [ ]:
# 單一參數掃描：max_depth 與 min_samples_leaf
scans = {
    "tree__max_depth": np.arange(1, 21),
    "tree__min_samples_leaf": np.array([1, 2, 3, 5, 8, 12, 20, 30, 50, 80, 120, 200]),
}
vc_res = {}
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for ax, (param, values) in zip(axes, scans.items()):
    tr_s, va_s = validation_curve(default_pipe, X_train, y_train, param_name=param, param_range=values, cv=cv, scoring="accuracy")
    vc_res[param] = pd.DataFrame({"train": tr_s.mean(1), "val": va_s.mean(1), "val_std": va_s.std(1)}, index=values)
    for name, s in [("train", tr_s), ("validation", va_s)]:
        ax.plot(values, s.mean(1), "o-", ms=3, label=name)
        ax.fill_between(values, s.mean(1) - s.std(1), s.mean(1) + s.std(1), alpha=0.2)
    ax.set_xlabel(param.split("__")[1])
    ax.set_ylabel("accuracy")
    ax.legend()
axes[1].set_xscale("log")
axes[0].set_title("Validation curve: max_depth (larger = more complex)")
axes[1].set_title("Validation curve: min_samples_leaf (larger = simpler)")
plt.show()
for param, tbl in vc_res.items():
    print(param)
    display(tbl.round(4).T)

In [ ]:
# 2D 搜尋：max_depth × min_samples_leaf
param_grid = {
    "tree__max_depth": [2, 3, 4, 5, 6, 8, 10, None],
    "tree__min_samples_leaf": [1, 3, 5, 10, 20, 40],
}
gs = GridSearchCV(default_pipe, param_grid, cv=cv, scoring="accuracy", n_jobs=-1)
gs.fit(X_train, y_train)
grid_res = pd.DataFrame(
    {
        "max_depth": [str(v) for v in gs.cv_results_["param_tree__max_depth"]],
        "min_samples_leaf": gs.cv_results_["param_tree__min_samples_leaf"].astype(int),
        "mean": gs.cv_results_["mean_test_score"],
        "std": gs.cv_results_["std_test_score"],
    }
)
print("best_params_：", gs.best_params_, "  CV accuracy：", round(gs.best_score_, 4), " 葉子數：", gs.best_estimator_["tree"].get_n_leaves())

heat = grid_res.pivot_table(index="max_depth", columns="min_samples_leaf", values="mean").reindex([str(v) for v in param_grid["tree__max_depth"]])
fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(heat, annot=True, fmt=".3f", cmap="viridis", ax=ax)
ax.set_title("CV accuracy (higher is better)")
plt.show()

In [ ]:
# TODO 17（問答）validation curve 與 2D 搜尋
#
# (1) max_depth 的 validation curve：
#     (a) val 最高在哪個深度？深度 3 到 8 之間的差距，跟 val_std 比起來大嗎？
#     (b) train 一路升到多少？過擬合從哪裡開始明顯（train − val 的差距變大）？
# 答：
#
# (2) min_samples_leaf 的 validation curve：
#     (a) 這條曲線的「複雜」在左邊還是右邊？為什麼？
#     (b) min_samples_leaf 很大（例如 200）時，train 和 val 一起掉，這是欠擬合還是過擬合？
#         樹最多還能有幾片葉子？（N ≈ 1047，每片至少 200 筆）
# 答：
#
# (3) heatmap：
#     (a) max_depth = None 那一列，min_samples_leaf 從 1 加到 20，CV accuracy 怎麼變？
#         為什麼「不限深度、但葉子至少 20 筆」也能和「深度 3」差不多好？兩者限制複雜度的方式有什麼不同？
#     (b) CV 最佳那一格，跟它周圍幾格差多少？跟 std 比呢？「最佳」可靠嗎？
# 答：
#
# (4) 第 5 節開頭的表：完全長滿的樹 log loss 約 8，比 Dummy prior 的 0.665 差十倍以上，深度 3 的樹卻只有 0.57。
#     (a) 長滿的樹的葉子大多只有 1–3 筆（實驗 2），predict_proba 大多是多少？
#     (b) 驗證資料裡一個人落到「生還率 0」的葉子、但他其實生還了，這一筆的 log loss 是多少？（第三章 TODO 2 的 log 0）
#     (c) 所以要用決策樹的機率（例如排序、算期望值），樹要怎麼長？
# 答：

In [ ]:
# 成本複雜度剪枝：先拿到一串 α，再用 CV 選
prep_full = make_prep().fit(X_train)
path = DecisionTreeClassifier(random_state=0).cost_complexity_pruning_path(prep_full.transform(X_train), y_train)
alphas = path.ccp_alphas[:-1]  # 最後一個 α 剪到只剩根節點，不列入
print(f"剪枝路徑上有 {len(alphas)} 個 α，範圍 {alphas.min():.2e} ~ {alphas.max():.2e}")

ccp_rows = []
for a in alphas:
    pipe_a = Pipeline([("prep", make_prep()), ("tree", DecisionTreeClassifier(random_state=0, ccp_alpha=a))])
    s = cross_val_score(pipe_a, X_train, y_train, cv=cv)
    n_leaves = pipe_a.fit(X_train, y_train)["tree"].get_n_leaves()
    ccp_rows.append({"alpha": a, "leaves": n_leaves, "mean": s.mean(), "std": s.std()})
ccp_res = pd.DataFrame(ccp_rows)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(ccp_res["alpha"], ccp_res["mean"], "o-", ms=3)
axes[0].fill_between(ccp_res["alpha"], ccp_res["mean"] - ccp_res["std"], ccp_res["mean"] + ccp_res["std"], alpha=0.2)
axes[0].set_xscale("symlog", linthresh=1e-4)
axes[0].set_xlabel("ccp_alpha (larger = simpler)")
axes[0].set_ylabel("CV accuracy")
axes[0].set_title("CV accuracy along the pruning path")
axes[1].plot(ccp_res["alpha"], ccp_res["leaves"], "o-", ms=3)
axes[1].set_xscale("symlog", linthresh=1e-4)
axes[1].set_yscale("log")
axes[1].set_xlabel("ccp_alpha")
axes[1].set_ylabel("leaves (log)")
axes[1].set_title("Tree size along the pruning path")
fig.tight_layout()
plt.show()
display(ccp_res.sort_values("mean", ascending=False).head(8).round(5))

#### 最小例子：TODO 18（剪枝路徑上的 1-SE rule）

5 列的假結果表，accuracy 越高越好；決策樹的「簡單」= $\alpha$ 大（葉子少）。

1. **正確答案**：逐列判斷，再挑 $\alpha$ 最大的。目標：`alpha = 0.008`（6 片葉子）。
2. **pandas 的積木**：門檻、布林遮罩；`idxmax` 拿到的是 index 標籤（不是位置）。
3. **留給你**：從 `ok` 裡挑出「最簡單」的一列。

In [ ]:
# 最小例子：TODO 18（剪枝路徑上的 1-SE rule）
res_mini = pd.DataFrame(
    {
        "alpha": [0.0, 0.001, 0.003, 0.008, 0.02],
        "leaves": [260, 60, 18, 6, 2],
        "mean": [0.760, 0.805, 0.815, 0.802, 0.775],
        "std": [0.03, 0.02, 0.015, 0.02, 0.02],
    }
)
display(res_mini)

# ---- 1. 正確答案：逐列判斷 ----
best_loop = max(res_mini.to_dict("records"), key=lambda r: r["mean"])
thr_loop = best_loop["mean"] - best_loop["std"]
ok_loop = []
for r in res_mini.to_dict("records"):
    print(f"alpha={r['alpha']:<6} leaves={r['leaves']:<4} acc={r['mean']:.3f} ≥ {thr_loop:.3f}？{r['mean'] >= thr_loop}")
    if r["mean"] >= thr_loop:
        ok_loop.append(r)
print("正確答案：", max(ok_loop, key=lambda r: r["alpha"]))

# ---- 2. pandas 的積木 ----
best_mini = res_mini.loc[res_mini["mean"].idxmax()]
print("\nbest = res.loc[res['mean'].idxmax()]：\n", best_mini.to_dict())
ok_mini = res_mini[res_mini["mean"] >= best_mini["mean"] - best_mini["std"]]
display(ok_mini)

# ---- 3. 留給你 ----
# 問題：從 ok_mini 挑「alpha 最大」的那一列，用 .loc[...idxmax()] 或 sort_values(...).iloc[0] 怎麼寫？

In [ ]:
# one-standard-error rule：跟 CV 最佳差不到一個標準差的 α 中，挑最大的（= 最簡單的樹）
best_ccp = ccp_res.loc[ccp_res["mean"].idxmax()]

# TODO 18（程式）
#   - ok：mean ≥ best_ccp["mean"] − best_ccp["std"] 的所有列
#   - choice_ccp：ok 裡 alpha 最大的一列（同 alpha 不會重複，不用再比其他欄）
raise NotImplementedError("TODO 18：1-SE")

print("CV 最佳 ：", best_ccp.round(5).to_dict())
print("1-SE 選擇：", choice_ccp.round(5).to_dict())
print("✅ 1-SE 的 alpha 應 ≥ CV 最佳的 alpha、葉子數應 ≤ CV 最佳：", choice_ccp["alpha"] >= best_ccp["alpha"] and choice_ccp["leaves"] <= best_ccp["leaves"])

from sklearn.base import clone

model_ccp_best = clone(default_pipe).set_params(tree__ccp_alpha=best_ccp["alpha"]).fit(X_train, y_train)
model_1se = clone(default_pipe).set_params(tree__ccp_alpha=choice_ccp["alpha"]).fit(X_train, y_train)
print("1-SE 的樹：深度", model_1se["tree"].get_depth(), "、葉子", model_1se["tree"].get_n_leaves())
print(export_text(model_1se["tree"], feature_names=FEATURES, show_weights=True))

In [ ]:
# 學習曲線：再多一點資料會更好嗎？（1-SE 的剪枝樹 vs 完全長滿的樹）
fig, axes = plt.subplots(1, 2, figsize=(13, 4), sharey=True)
lc_tbls = {}
for ax, (name, model) in zip(axes, [("1-SE pruned tree", model_1se), ("fully grown tree", default_pipe)]):
    sizes, tr_s, va_s = learning_curve(model, X_train, y_train, cv=cv, scoring="accuracy", train_sizes=np.linspace(0.1, 1.0, 8))
    for lab, s in [("train", tr_s), ("validation", va_s)]:
        ax.plot(sizes, s.mean(1), "o-", label=lab)
        ax.fill_between(sizes, s.mean(1) - s.std(1), s.mean(1) + s.std(1), alpha=0.2)
    ax.set_xlabel("training samples")
    ax.set_title(f"Learning curve: {name}")
    ax.legend()
    lc_tbls[name] = pd.DataFrame({"n": sizes, "train": tr_s.mean(1), "val": va_s.mean(1), "val_std": va_s.std(1)}).round(4)
axes[0].set_ylabel("accuracy")
plt.show()
for name, tbl in lc_tbls.items():
    print(name)
    display(tbl.T)

In [ ]:
# TODO 19（問答）剪枝與學習曲線
#
# (1) 剪枝路徑：
#     (a) α = 0 時是哪一棵樹？最大的 α 時剩幾片葉子？
#     (b) CV 最佳的 α 有幾片葉子？1-SE 選的呢？1-SE 門檻是多少（寫出 mean − std 的算式）？
#     (c) 1-SE 的樹印出來之後，和第 4 節深度 3 的樹比，像不像？
# 答：
#
# (2) 「先長滿再剪」（ccp_alpha）和「一開始就不要長太深」（max_depth、min_samples_leaf，叫做 pre-pruning）有什麼不同？
#     提示：TODO 12 (2) —— 有時候某一刀本身沒什麼進步，但它下面的那一刀很好。
#     pre-pruning 在那一刀就停了，會發生什麼事？post-pruning 呢？
# 答：
#
# (3) 學習曲線：
#     (a) 兩張圖的 train 線各在哪裡？長滿的樹 train 為什麼這麼高？
#     (b) 兩張圖的 val 最後幾點還在上升嗎？哪一個模型「多給資料」比較有幫助？
#     (c) 剪枝樹 train 和 val 已經很接近、而且都停在約 0.8：瓶頸是資料量、模型容量，還是特徵？
# 答：

---
## 6. 測試集評估

模型都已經用 CV 決定好，測試集只評估一次。除了決策樹，另外放兩個前幾章的模型當對照（都沒有調參，只是參考）：

- **邏輯回歸**（第三章）：數值欄標準化 + one-hot，$C = 1$。
- **KNN**（第四章）：數值欄標準化 + one-hot，$k = 15$。這份資料的特徵單位不同（年齡 vs 票價 vs 0／1），依第四章的結論必須標準化。

#### 指標

- **accuracy、precision、recall、F1**：二元分類，以「生還 = 1」為正類。recall = 真的生還的人裡，被預測生還的比例。
- **ROC AUC**：用預測機率排序；決策樹的機率只有幾種值（TODO 16 (2)），AUC 會吃虧。
- `log_result`／`compare`：把成績寫進 `results/titanic.csv`，並依指定指標排序。

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from ml_dl_llm.metrics import classification_metrics
from ml_dl_llm.results import compare, log_result

BENCH = "titanic"


def make_scaled_prep() -> ColumnTransformer:
    """給邏輯回歸、KNN 用：數值欄補值 + 標準化，類別欄 one-hot。"""
    return ColumnTransformer(
        [
            ("num", Pipeline([("imp", SimpleImputer(strategy="median")), ("sc", StandardScaler())]), NUM),
            ("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent")), ("oh", OneHotEncoder(drop="if_binary"))]), ["sex", "embarked"]),
        ]
    )


dummy = DummyClassifier(strategy="prior").fit(Z_train, y_train)
logreg = Pipeline([("prep", make_scaled_prep()), ("lr", LogisticRegression(max_iter=2000))]).fit(X_train, y_train)
knn = Pipeline([("prep", make_scaled_prep()), ("knn", KNeighborsClassifier(n_neighbors=15))]).fit(X_train, y_train)
scratch_full = DecisionTree().fit(Xtr, ytr)
default_pipe.fit(X_train, y_train)
bp = {k.split("__")[-1]: v for k, v in gs.best_params_.items()}

# 名稱: (預測, 機率或 None, 要記下的參數)
eval_models = {
    "Dummy prior": (dummy.predict(Z_test), dummy.predict_proba(Z_test)[:, 1], {}),
    "Sex rule": (SexRule().fit(Z_train, y_train).predict(Z_test), None, {}),
    "Tree depth 3 (scratch)": (scratch_d3.predict(Xte), scratch_d3.predict_proba(Xte)[:, 1], {"max_depth": 3}),
    "Tree fully grown (scratch)": (scratch_full.predict(Xte), scratch_full.predict_proba(Xte)[:, 1], {"max_depth": None}),
    "Tree depth 3 (sklearn)": (tree_pipe.predict(X_test), tree_pipe.predict_proba(X_test)[:, 1], {"max_depth": 3}),
    "Tree fully grown (sklearn)": (default_pipe.predict(X_test), default_pipe.predict_proba(X_test)[:, 1], {}),
    "Tree grid best (sklearn)": (gs.best_estimator_.predict(X_test), gs.best_estimator_.predict_proba(X_test)[:, 1], bp),
    "Tree ccp CV-best (sklearn)": (model_ccp_best.predict(X_test), model_ccp_best.predict_proba(X_test)[:, 1], {"ccp_alpha": round(float(best_ccp["alpha"]), 6)}),
    "Tree ccp 1-SE (sklearn)": (model_1se.predict(X_test), model_1se.predict_proba(X_test)[:, 1], {"ccp_alpha": round(float(choice_ccp["alpha"]), 6)}),
    "LogReg C=1 (sklearn, ref)": (logreg.predict(X_test), logreg.predict_proba(X_test)[:, 1], {"C": 1.0}),
    "KNN k=15 (sklearn, ref)": (knn.predict(X_test), knn.predict_proba(X_test)[:, 1], {"k": 15}),
}

test_scores = {}
for name, (pred, prob, params) in eval_models.items():
    m = classification_metrics(y_test, pred, prob)
    test_scores[name] = m
    log_result(BENCH, name, m, **params)

display(pd.DataFrame(test_scores).T)
compare(BENCH, "accuracy")

---
## 7. 診斷

決策樹的優點之一是**可以用規則解釋**：每個預測都能說出「因為他是男性、三等艙、9.5 歲以上……」。這一節看它錯在哪裡，以及特徵重要度可不可信。

#### 本節名詞

- **混淆矩陣**：列 = 真實（0 罹難、1 生還）、欄 = 預測。
- **分組 accuracy**：依 sex × pclass 分成 6 組，各算 accuracy 與錯誤筆數。
- **純雜訊特徵**：加一欄跟生還完全無關的隨機連續數字 `noise`，看重要度會不會被騙。
- **permutation importance**（`sklearn.inspection.permutation_importance`）：在**測試集**上把某一欄打亂（其他欄不動），看 accuracy 掉多少；重複 30 次取平均。跟 impurity-based 不同，它看的是「沒見過的資料上，少了這一欄的資訊會怎樣」。

In [ ]:
from ml_dl_llm.metrics import plot_confusion_matrix

final = model_1se
pred_te = final.predict(X_test)
proba_te = final.predict_proba(X_test)[:, 1]
plot_confusion_matrix(y_test, pred_te, labels=["died", "survived"])
plt.title("1-SE pruned tree on test")
plt.show()

diag = X_test.assign(survived=y_test, pred=pred_te, proba=proba_te, correct=pred_te == yte)
group_tbl = diag.groupby(["sex", "pclass"], observed=True).agg(
    n=("correct", "size"), accuracy=("correct", "mean"), errors=("correct", lambda c: int((~c).sum())),
    true_survival=("survived", "mean"), mean_pred_proba=("proba", "mean"),
)
display(group_tbl.round(3))
print(f"測試集錯 {int((~diag['correct']).sum())} 筆 / {len(diag)}")

# 錯誤樣本落在哪些葉子？那些葉子的生還率是多少？
leaf_te = final["tree"].apply(final["prep"].transform(X_test))
rules_1se = leaf_rules(final["tree"].tree_, FEATURES)
err_leaf = pd.Series(leaf_te[~diag["correct"].to_numpy()]).value_counts().rename("test errors")
display(rules_1se.join(err_leaf).fillna({"test errors": 0}).sort_values("test errors", ascending=False).round(3))

In [ ]:
# 特徵重要度可信嗎？加一欄純雜訊
from sklearn.inspection import permutation_importance

rng = np.random.default_rng(1)
Zn_train = Z_train.assign(noise=rng.random(len(Z_train)))
Zn_test = Z_test.assign(noise=rng.random(len(Z_test)))

imp_rows = {}
for name, est in [
    ("fully grown", DecisionTreeClassifier(random_state=0)),
    ("1-SE pruned", DecisionTreeClassifier(random_state=0, ccp_alpha=choice_ccp["alpha"])),
]:
    est.fit(Zn_train, y_train)
    perm = permutation_importance(est, Zn_test, y_test, n_repeats=30, random_state=0, scoring="accuracy")
    imp_rows[(name, "impurity (train)")] = pd.Series(est.feature_importances_, index=Zn_train.columns)
    imp_rows[(name, "permutation (test)")] = pd.Series(perm.importances_mean, index=Zn_train.columns)
imp_cmp = pd.DataFrame(imp_rows)
display(imp_cmp.round(4))

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for ax, name in zip(axes, ["fully grown", "1-SE pruned"]):
    imp_cmp[name].sort_values("impurity (train)").plot.barh(ax=ax)
    ax.set_title(f"{name}: impurity vs permutation importance")
fig.tight_layout()
plt.show()

In [ ]:
# TODO 20（問答，本章總結）
#
# (1) 錯在哪裡：
#     (a) 混淆矩陣：錯誤主要是「把生還的人猜成罹難」還是反過來？哪一種多？
#     (b) 分組表：哪兩組錯最多？各自的真實生還率是多少？模型對那一組幾乎都猜同一個答案嗎？
#         如果某一片葉子的訓練生還率剛好 0.5（template 測試時，女性三等艙那一片是 89 : 89），模型猜什麼？為什麼？（DecisionTree.predict 的 docstring）
#     (c) 如果一個葉子的生還率是 0.4，裡面每個人都會被猜「罹難」，所以 40% 的人一定錯。
#         這種錯誤是「模型不夠好」，還是「這 9 個特徵本來就分不開這群人」？要怎麼分辨？
#         （提示：TODO 11 (2) 特徵完全相同、標籤不同的人；以及有沒有別的資訊（例如 name 裡的稱謂 Master／Miss、家庭成員）可以用）
# 答：
#
# (2) 特徵重要度：
#     (a) 完全長滿的樹，noise 的 impurity importance 排第幾？大約多少？permutation importance 呢？
#     (b) 為什麼 impurity importance 會被騙？（TODO 16 (3)(b)：很多候選門檻、用訓練資料算；
#         長滿的樹在很深、很小的節點上「碰巧」用 noise 切掉幾個人）
#     (c) 剪枝之後，noise 的 impurity importance 怎麼變？為什麼？
#     (d) 結論：要回答「哪個特徵重要」時，你會用哪一種？
# 答：
#
# (3) 跟前幾章比（看第 6 節的表）：
#     (a) 剪枝樹、邏輯回歸、KNN 的 test accuracy 各是多少？262 筆的測試集（1 筆 ≈ 0.38%）上差幾筆？
#         能說誰比較好嗎？
#     (b) 比較表（填空）：
#
#         |                      | 決策樹 | 邏輯回歸 | KNN |
#         | 需要標準化嗎         |        |          |     |
#         | 決策邊界的形狀       |        |          |     |
#         | 交互作用（sex × age）|        |          |     |
#         | 怎麼解釋一個預測     |        |          |     |
#         | 對訓練資料的敏感度   |        |          |     |
#         | 機率的品質           |        |          |     |
#
#     (c) 決策樹最大的缺點是哪一個？第 10 章 Random Forest、第 12 章 GBDT 各用什麼方法補救？
# 答：

---
## 8. 本章小結

（完成所有 TODO、並經過回饋與修正之後，依 README「章末小結」填寫：問答註解轉成 markdown、結果表、核心觀念、待補強。）

### 方法與結果

Titanic（train / test 依載入結果；CV = 5-fold 分層）。

| 方法 | 做法 | 關鍵性質 | CV accuracy | test accuracy |
| --- | --- | --- | --- | --- |
| Dummy prior | 輸出生還比例 | | | |
| 性別規則 | 女性生還、男性罹難 | | | |
| 決策樹 深度 3（手寫） | 排序 + cumsum 找切點、遞迴長樹 | | | |
| 決策樹 完全長滿 | | | | |
| 決策樹 grid 最佳 | max_depth × min_samples_leaf | | | |
| 決策樹 ccp 1-SE | 成本複雜度剪枝 | | | |
| 邏輯回歸（對照） | 標準化 + $C = 1$ | | — | |
| KNN（對照） | 標準化 + $k = 15$ | | — | |

### 核心觀念

1.
2.
3.

### 待補強（之後回來可以學）

**分裂與不純度**
- ☐ ID3、C4.5 的細節：多元分裂、增益比（gain ratio）為什麼能修正「偏好多值特徵」
- ☐ 類別特徵的最佳二元分裂：$K$ 個類別依「生還率」排序後只需試 $K - 1$ 種切法（二元分類時是最佳的）
- ☐ 缺值的處理：C4.5 的分數權重、CART 的 surrogate split、sklearn ≥ 1.3 原生支援 NaN
- ☐ 回歸樹（`DecisionTreeRegressor`）：不純度換成變異數（MSE），葉子預測平均值

**剪枝與複雜度**
- ☐ 成本複雜度剪枝的 weakest-link 演算法（`cost_complexity_pruning_path` 怎麼算出那一串 α）
- ☐ `max_leaf_nodes` 的 best-first 長法；`min_impurity_decrease`
- ☐ 樹的 VC 維度與泛化界

**特徵與解釋**
- ☐ 特徵工程：從 name 取出稱謂（Mr／Mrs／Miss／Master）、家庭人數、cabin 的甲板字母、是否有 cabin
- ☐ impurity importance 的偏差（Strobl et al. 2007）；permutation importance 在特徵相關時的問題；SHAP（TreeSHAP）
- ☐ 斜切樹（oblique tree）：每一刀用多個特徵的線性組合

**延伸模型**
- ☐ Random Forest（第 10 章）、GBDT（第 12 章）：用很多棵樹降低變異／偏差
- ☐ 模型比較的統計檢定：McNemar 檢定、bootstrap 信賴區間（小測試集上差幾筆算不算數）